# Full-Arm Manipulation Planner

Loads the full Panda arm + hand + bottle-cap + obstacle scene (same setup as
`full_arm_nhr.ipynb`), generates a C-free cover with Drake's **native**
IRIS-ZO / clique-cover implementation (unmodified - no custom Python
reimplementation from #69-71), and plans a trajectory with `ManipulationPlanner`,
using the grasp polytope derived from the wrist-axis analysis (#80, see
`algorithms/nlp_sampling/tc_space_investigation_summary.md` Part 2) instead of
a hand-picked box.

This is a first-pass scaffold: sections 1-2 have been run and validated
end-to-end; section 4's IRIS-ZO clique-cover + MOSEK MIQP solve is expensive
(minutes+, needs a MOSEK license) and has **not** been executed as part of
writing this notebook - see the caveats in section 4 before running it.

In [1]:
# Cell 1: Import necessary libraries

import os
import numpy as np
import time
from pydrake.all import (
    DiagramBuilder, AddMultibodyPlantSceneGraph, Parser,
    Role, MeshcatVisualizer, StartMeshcat, RationalForwardKinematics, CspaceFreePolytope,
    SeparatingPlaneOrder, Rgba, RollPitchYaw,
    LinearEqualityConstraint, Sphere, Parallelism, AddDefaultVisualization,
    ConnectPlanarSceneGraphVisualizer, IrisFromCliqueCoverOptions,
    CommonSampledIrisOptions, IrisZoOptions, IrisNp2Options,
    IrisInConfigurationSpaceFromCliqueCover, RandomGenerator, RobotDiagramBuilder,
    SceneGraphCollisionChecker, MultibodyPlant, SceneGraph,
    SolverOptions, CommonSolverOption, GeometrySet, ScsSolver
)
from pydrake.geometry.optimization import GraphOfConvexSetsOptions, HPolyhedron, VPolytope, Point, Hyperellipsoid
from pydrake.geometry.optimization import ConvexHull as DrakeConvexHull
from pydrake.planning import GcsTrajectoryOptimization
from pydrake.solvers import MathematicalProgram, Solve, MosekSolver
from pydrake.trajectories import CompositeTrajectory
from pydrake.common import FindResourceOrThrow
from scipy.spatial import ConvexHull
from pydrake.math import RigidTransform, RotationMatrix
from pydrake.multibody.inverse_kinematics import InverseKinematics
from pydrake.solvers import Solve, SnoptSolver, IpoptSolver, OsqpSolver, ScsSolver
from functools import partial
from dataclasses import dataclass
import matplotlib.pyplot as plt
from ipywidgets import widgets

from pathlib import Path
import sys

def _find_repo_root(start: Path, markers=(".git", "README.md")) -> Path:
    for candidate in (start, *start.parents):
        if any((candidate / m).exists() for m in markers):
            return candidate
    raise RuntimeError(f"Could not locate repo root above {start}")

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (REPO_ROOT, REPO_ROOT / "algorithms", REPO_ROOT / "algorithms" / "nlp_sampling"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from ciris_plant_visualizer import CIrisPlantVisualizer

In [ ]:
# MOSEK licence. MOSEK reads its path from MOSEKLM_LICENSE_FILE, which only the conda env's activate script
# sets; without it Drake silently falls back to another solver. Holding one licence for the whole session
# also avoids a checkout per solve: the A* graph cells below take ~1 s instead of ~50-90 s.
os.environ.setdefault("MOSEKLM_LICENSE_FILE", str(REPO_ROOT / "mosek.lic"))
mosek_licence = MosekSolver.AcquireLicense() if MosekSolver().enabled() else None
print("MOSEK enabled:", MosekSolver().enabled())

## 1. Load the full Panda arm + cap + obstacles

Verbatim from `full_arm_nhr.ipynb` (same URDFs, same welds, same obstacle placement) - this is what makes the wrist-axis polytope built in section 2 (from a *separate*, obstacle-free plant in `wrist_axis_grasp_polytope.py`) valid here: same URDF load order means the same position indexing.

In [2]:
print("Setting up the plant and scene graph...")

builder = RobotDiagramBuilder(time_step=0.0)
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = Parser(plant, scene_graph)
parser.SetAutoRenaming(True)

Setting up the plant and scene graph...


In [3]:
print("Loading Panda robot models...")

# panda_arm  = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_arm.urdf")[0]
panda_arm = parser.AddModels(url="file:///home/julialopezgomez/optimisation-based-manipulation-planner/my_sdfs/panda_arm.urdf")[0]
panda_hand = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_hand.urdf")[0]

print("Welding Panda arm and hand...")

plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("panda_link0", panda_arm),
    RigidTransform())

X_8H = RigidTransform(RollPitchYaw(0.0, 0.0, -np.deg2rad(45.0)), [0.0, 0.0, 0.0])
plant.WeldFrames(
    plant.GetFrameByName("panda_link8", panda_arm),
    plant.GetFrameByName("panda_hand", panda_hand),
    X_8H)

print("Setting default finger joint positions...")
plant.GetJointByName("panda_finger_joint1", panda_hand).set_default_translation(-0.024)

Loading Panda robot models...
Welding Panda arm and hand...
Setting default finger joint positions...


In [4]:
print("Adding the bottle cap and obstacles...")

cap = parser.AddModels(file_name="my_sdfs/bottle_cap.sdf")[0]
obstacle1 = parser.AddModels("my_sdfs/obstacle.sdf")[0]
obstacle3 = parser.AddModels("my_sdfs/obstacle.sdf")[0]

plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("base_link", cap),
    RigidTransform(RotationMatrix(), [0.5, 0, 0]))

obstacle_pose1 = RigidTransform(RotationMatrix(), [0.51, 0.031, 0.01])
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle1),
    obstacle_pose1)

obstacle_pose3 = RigidTransform(RotationMatrix(), [0.467, -0.005, 0.01])
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle3),
    obstacle_pose3)

Adding the bottle cap and obstacles...


<WeldJoint name='world_welds_to_obstacle_link' index=16 model_instance=6>

In [5]:
print("Finalising plant...")
print("Bodies:", plant.num_bodies(), "Joints:", plant.num_joints())


plant.Finalize()

inspector = scene_graph.model_inspector()
num_frames = len(list(inspector.GetAllFrameIds()))
num_geoms = sum(inspector.NumGeometriesForFrame(fid) for fid in inspector.GetAllFrameIds())
print("Frames:", num_frames, "Geometries:", num_geoms)


print("Plant finalised.")

print("Number of positions: ", plant.num_positions())


q_star = np.zeros(plant.num_positions())

# print("Initialising CspaceFreePolytope...")
# cspace_free_polytope = CspaceFreePolytope(
#     plant,
#     scene_graph,
#     SeparatingPlaneOrder.kAffine,
#     q_star)

print("Initializing CIrisPlantVisualizer...")
visualizer = CIrisPlantVisualizer(
    plant,
    builder,
    scene_graph,
    # cspace_free_polytope,
    viz_role=Role.kIllustration,
    allow_plus_3dof=True
)

visualizer.task_space_diagram.ForcedPublish(visualizer.task_space_diagram_context)

INFO:drake:Meshcat listening for connections at http://localhost:7002


Finalising plant...
Bodies: 18 Joints: 17
Frames: 18 Geometries: 94
Plant finalised.
Number of positions:  10
Initializing CIrisPlantVisualizer...
Visualisations won't work properly. Can't visualize the TC-Space of plants with more than 3-DOF. The first 3 DOF from the plant will be visualized


In [7]:
sliders = []

plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

for i in range(plant.num_positions()):
    q_low = plant.GetPositionLowerLimits()[i]
    q_high = plant.GetPositionUpperLimits()[i]
    step = (q_high - q_low) / 100
    sliders.append(widgets.FloatSlider(
        min=q_low, max=q_high, 
        value=0, step=step, 
        description=f"q{i}"))
    
q = np.zeros(plant.num_positions())

def report_collisions(q):
    plant.SetPositions(plant_context, q)
    query_object = scene_graph.get_query_output_port().Eval(
        scene_graph.GetMyContextFromRoot(diagram_context)
    )
    inspector = scene_graph.model_inspector()

    pairs = query_object.ComputePointPairPenetration()
    if not pairs:
        print("No penetrations")
        return

    for pair in pairs:
        name_A = inspector.GetName(pair.id_A)
        name_B = inspector.GetName(pair.id_B)
        print(f"{name_A} <-> {name_B}, depth={pair.depth}")



out = widgets.Output()
display(out)

def handle_slider_change(change, idx):
    q[idx] = change["new"]
    plant.SetPositions(plant_context, q)
    diagram.ForcedPublish(diagram_context)
    with out:
        out.clear_output(wait=True)
        
        report_collisions(q)
        print(f"{visualizer.check_collision_q_by_ik(q)} \t {q}", flush=True)
    
    
idx = 0
for slider in sliders:
    slider.observe(partial(handle_slider_change, idx = idx), names='value')
    idx+=1

for slider in sliders:
    display(slider)

Output()

FloatSlider(value=0.0, description='q0', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=0.0, description='q1', max=1.7628, min=-1.7628, step=0.035255999999999996)

FloatSlider(value=0.0, description='q2', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=-0.0698, description='q3', max=-0.0698, min=-3.0718, step=0.03002)

...

FloatSlider(value=0.0, description='q5', max=3.7525, min=-0.0175, step=0.0377)

FloatSlider(value=0.0, description='q6', max=1.0, min=-1.0, step=0.02)

FloatSlider(value=0.0, description='q7', max=0.0, min=-0.045, step=0.00045)

FloatSlider(value=0.0, description='q8', max=0.045, step=0.00045)

FloatSlider(value=0.0, description='q9', max=3.14, min=-3.14, step=0.06280000000000001)

In [8]:
def joint_position_index(plant, joint_name, model_instance=None):
    if model_instance is None:
        joint = plant.GetJointByName(joint_name)
    else:
        joint = plant.GetJointByName(joint_name, model_instance)

    if joint.num_positions() != 1:
        raise ValueError(
            f"Joint {joint_name} has {joint.num_positions()} positions; "
            "this helper assumes a 1-DOF joint."
        )

    return joint.position_start()

for i, name in enumerate(plant.GetPositionNames()):
    print(f"{i:2d}: {name}")

 0: panda_panda_joint1_q
 1: panda_panda_joint2_q
 2: panda_panda_joint3_q
 3: panda_panda_joint4_q
 4: panda_panda_joint5_q
 5: panda_panda_joint6_q
 6: panda_panda_joint7_q
 7: panda_hand_panda_finger_joint1_x
 8: panda_hand_panda_finger_joint2_x
 9: bottle_cap_cap_to_base_q


## 2. Grasp polytope from the wrist-axis analysis (#80)

Reuses `wrist_axis_grasp_polytope.py`'s already-validated construction directly
(see `tc_space_investigation_summary.md` Part 2 for the full derivation and
plots): the wrist joint's rotation axis passes through the end-effector
frame's own origin, so the grasp constraint is *exactly* (not approximately)
invariant to it - the feasible region is the wrist's full range times a
small `+-0.001` rad margin on the other 6 arm joints, fingers pinned at
their found grasp width.

That construction runs on its own separate, obstacle-free plant
(`nlp_sampling_standalone_test.build_minimal_plant()`), built from the same
URDFs in the same order as section 1's plant - the index-parity asserts
below confirm the two plants agree on position ordering before any array
built on one is reused on the other.

Composed here into two full 10-dim boxes over *this* plant's actual
configuration space (`ManipulationPlanner` needs `CP`/`CG` in the same
ambient dimension as the `cs_free` regions IRIS-ZO will generate in
section 4, not just the 4 "gripper" dims):

- **`grasp_polytope`** - the wrist-axis box on the 6 arm joints + wrist,
  the fingers' own physical grasp tolerance (`[-0.025,-0.024]`/
  `[0.024,0.025]`, the same constants used everywhere else in this project -
  not q0-specific), and the cap left **free** over its full range. The cap
  can't be pinned here: it's the DOF that actually advances between
  grasp/release cycles in the MIQP below (`ManipulationPlanner`'s
  constraints 3/4 tie *consecutive* grasp/release cap values together, not
  every point to one fixed value) - pinning it would force every grasp/
  release knot point to the same cap angle and make the problem trivial or
  infeasible.
- **`placement_polytope`** - the same convention `full_arm_nhr.ipynb` used
  (wrist full range, a wider/more-open finger range, cap free), lifted from
  4D to the full 10D space by leaving the 6 arm joints unconstrained
  (placement was never about a specific arm posture, only the gripper
  end-state). By construction `grasp_polytope` is a strict subset of
  `placement_polytope` here.

### Find grasp configuration

We will build the grasp polytope around a specific grasp configuration, which we find by running the nlp sampling downhill phase 1

In [9]:
from wrist_axis_grasp_polytope import find_grasp_seed
from nlp_sampling_standalone_test import make_grasp_constraints, build_autodiff_companion, make_joint_indices, make_joint_limits

plant, plant_context = visualizer.plant, visualizer.plant_context
E = plant.GetFrameByName("panda_hand", panda_hand)
Cap = plant.GetFrameByName("base_link", cap)
plant_ad, plant_context_ad, E_ad, Cap_ad = build_autodiff_companion(plant, E, Cap)
h_grasp, g_grasp, Jh, Jg = make_grasp_constraints(
    plant, plant_context, 
    plant_ad, plant_context_ad, 
    E, Cap,
    E_ad, Cap_ad)
idx = make_joint_indices(plant, panda_arm, panda_hand, cap)
idx_wrist, idx_f1, idx_f2, idx_cap = idx["idx_wrist"], idx["idx_f1"], idx["idx_f2"], idx["idx_cap"]
lower, upper = make_joint_limits(plant, idx_wrist, idx_f1, idx_f2) # grasp limits

grasp_constraints = dict(
    plant=plant,
    h=h_grasp,
    g=g_grasp,
    Jh=Jh,
    Jg=Jg,
    idx_wrist=idx_wrist,
    lower=lower,
    upper=upper,
    _diagram=visualizer.task_space_diagram,
    _diagram_context=visualizer.task_space_diagram_context,
    _plant_context=plant_context,
    _plant_context_ad=plant_context_ad
)

q0 = find_grasp_seed(grasp_constraints)
q0[idx_wrist] = -1.0
q0[idx_f1] = -0.02405
q0[idx_f2] = 0.02405
print(q0)

[-1.7374666  -1.65251925  1.80953996 -2.22679311  1.78239145  1.70796246
 -1.         -0.02405     0.02405     0.94488425]


In [10]:
plant.SetPositions(plant_context, q0)
diagram.ForcedPublish(diagram_context)
print("in collision:", visualizer.check_collision_q_by_ik(q0))

in collision: 0.0


### Build grasp polytope in the nullspace of the grasp constraint

In [11]:

g0 = grasp_constraints["g"](q0)
h0 = grasp_constraints["h"](q0)
Jh0 = grasp_constraints["Jh"](q0)
Jg0 = grasp_constraints["Jg"](q0)
# Jhg = np.vstack([Jh0] + [Jg0])

def nullspace_from_jacobian(J, rtol=1e-8):
    U, s, Vt = np.linalg.svd(J, full_matrices=True)

    tol = rtol * s[0] if len(s) else rtol
    rank = np.sum(s > tol)

    N = Vt[rank:, :].T       # (n, n-rank)
    P = N @ N.T              # (n, n)

    return N, P, s, rank

N, P, singular_values, rank = nullspace_from_jacobian(Jh0)

print("Jh:", Jh0.shape)
print("rank:", rank)
print("N:", N.shape)
print("P:", P.shape)

Jh: (4, 10)
rank: 4
N: (10, 6)
P: (10, 10)


In [12]:
margin = 1e-3

nq = plant.num_positions()

# ------------------------------------------------------------------
# 1. Turn h(q) = 0 into |h(q)| <= margin
#
# Linearisation around q0:
#
#   h(q) ≈ h0 + Jh0 @ (q - q0)
#
# and require:
#
#   -margin <= h(q) <= margin
#
# This produces 2 inequalities per equality constraint:
#
#   1. Jh0 @ q <= margin - h0 + Jh0 @ q0
#   2. -Jh0 @ q <= margin + h0 - Jh0 @ q0
# ------------------------------------------------------------------

eps_h = margin * np.ones_like(h0)

A_h = np.vstack([
    Jh0,
    -Jh0,
])

b_h = np.concatenate([
    eps_h - h0 + Jh0 @ q0,
    eps_h + h0 - Jh0 @ q0,
])

# ------------------------------------------------------------------
# 2. Linearise the grasp inequalities g(q) <= 0 + margin
#
#   g(q) ≈ g0 + Jg0 @ (q - q0) <= 0 + margin
#
# => Jg0 @ q <= margin - g0 + Jg0 @ q0
# ------------------------------------------------------------------

A_g = Jg0

b_g = margin - g0 + Jg0 @ q0

# ------------------------------------------------------------------
# 3. Add joint limits exactly
#
#   lower <= q <= upper
# ------------------------------------------------------------------

lower = np.asarray(grasp_constraints["lower"], dtype=float)
upper = np.asarray(grasp_constraints["upper"], dtype=float)

A_limits = np.vstack([
    np.eye(nq),
    -np.eye(nq),
])

b_limits = np.concatenate([
    upper,
    -lower,
])


# ------------------------------------------------------------------
# 4. Combine everything
#
# Final set:
#
#   P = {q in R^10 | A q <= b}
# ------------------------------------------------------------------

A = np.vstack([
    A_h,
    A_g,
    A_limits,
])

b = np.concatenate([
    b_h,
    b_g,
    b_limits,
])

grasp_polytope = HPolyhedron(A, b)

# ------------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------------

print("A shape:", A.shape)
print("b shape:", b.shape)
print("Polytope ambient dimension:", A.shape[1])
print("q0 inside polytope:", grasp_polytope.PointInSet(q0))

print("\nh(q0):", h0)
print("g(q0):", g0)


A shape: (31, 10)
b shape: (31,)
Polytope ambient dimension: 10
q0 inside polytope: True

h(q0): [-1.28691283e-05 -1.01563271e-04 -4.77217183e-04 -9.20447251e-05]
g(q0): [ 0.0009401  -0.0059401  -0.09999988]


In [13]:
# zero_motion = np.zeros(plant.num_positions())
# independent_joint_motions = np.eye(plant.num_positions())

# for i in range(plant.num_positions()):
#     delta_q_P = P @ independent_joint_motions[:, i]
#     print(f"\ndelta_q for joint {i}:", delta_q_P)
#     print(f"Jh0 @ delta_q_P for joint {i}:", Jh0 @ delta_q_P, "\t All close to zero:", np.allclose(Jh0 @ delta_q_P, np.zeros_like(Jh0 @ delta_q_P), atol=1e-8))
    
    # delta_q_Jh = Jh0 @ independent_joint_motions[:, i]
    # print(f"Jh0 @ delta_q_P for joint {i}:", delta_q_Jh)

In [14]:

placement_lower = plant.GetPositionLowerLimits().copy()
placement_upper = plant.GetPositionUpperLimits().copy()
placement_lower[idx_wrist], placement_upper[idx_wrist] = -1.0, 1.0
placement_lower[idx_f1], placement_upper[idx_f1] = -0.055, -0.024
placement_lower[idx_f2], placement_upper[idx_f2] = 0.024, 0.055
placement_polytope = HPolyhedron.MakeBox(placement_lower, placement_upper)

print(f"grasp_polytope: ambient_dimension={grasp_polytope.ambient_dimension()}")
print(f"placement_polytope: ambient_dimension={placement_polytope.ambient_dimension()}")
print("q0 in placement:", placement_polytope.PointInSet(q0), " q0 in grasp:", grasp_polytope.PointInSet(q0))

grasp_polytope: ambient_dimension=10
placement_polytope: ambient_dimension=10
q0 in placement: True  q0 in grasp: True


## 3. Manipulation Planner class

Copied verbatim from `algorithms/manipulation_planner/manipulation_planner.py`
(unmodified - same pattern as `manipulation_planner_3dof.ipynb`/
`manipulation_planner_4dof.ipynb`, which inline the class in a cell rather
than importing it). Its `_generate_cfree` is the "IRIS-ZO original Drake
implementation" step: `IrisZoOptions` + `IrisInConfigurationSpaceFromCliqueCover`
straight from `pydrake`, not this project's Python reimplementation from
#69-71.

References `visualizer`, `panda_arm`, `panda_hand`, `cap` as plain globals
(inside `display_trajectory`) - already defined by section 1, so no
module-global wiring is needed here the way the imported version needs
(compare `full_arm_nhr.ipynb`'s `_manipulation_planner_module.visualizer = visualizer` cell).

In [15]:
def convex_hull_to_hpolyhedron(hull):
    A = hull.equations[:, :-1]
    b = -hull.equations[:, -1]
    return HPolyhedron(A, b)


class ManipulationPlanner():

    def __init__(self,
            visualizer: CIrisPlantVisualizer,
            CP: HPolyhedron,
            CG: HPolyhedron,
            max_grasps: int = 20,
            model_instances: list = None,
            gripper_dim: int = None,
            cs_free: list[HPolyhedron] = None,
            irisalg: str = "irisnp"
        ):
        self.builder = visualizer.builder
        self.plant = visualizer.plant
        self.plant_context = visualizer.plant_context
        self.scene_graph = visualizer.scene_graph
        self.q_star = visualizer.q_star
        self.rat_fk = visualizer.rat_forward_kin
        self.inspector = visualizer.model_inspector
        self.diagram = visualizer.task_space_diagram
        self.diagram_context = visualizer.task_space_diagram_context
        self.lower_joint_limits = visualizer.q_lower_limits
        self.upper_joint_limits = visualizer.q_upper_limits
        self.visualize_cspace = visualizer.visualize_collision_constraint
        self.cspace_free_polytope = visualizer.cspace_free_polytope

        self.CP = CP
        self.CG = CG
        self.max_grasps = max_grasps

        self.model_instances = model_instances if model_instances is not None else [self.plant.GetModelInstanceByName("robot")]

        self.q_dim = self.plant.num_positions()
        self.gripper_dim = gripper_dim if gripper_dim is not None else self.q_dim


        self.cs_free = cs_free if cs_free is not None else self._generate_cfree(irisalg=irisalg)

    def set_max_grasps(self, max_grasps: int):
        self.max_grasps = max_grasps


    def compute_trajectory(self, x_init, x_goal, display=True):
        print("Finding path for minimum grasps")
        path = self._find_minimum_grasp_path(
                x_init = x_init,
                x_goal = x_goal,
                P = self.CP,
                G = self.CG,
                lower_joint_limits = self.lower_joint_limits,
                upper_joint_limits = self.upper_joint_limits,
                c_free_polytopes = self.cs_free
            )

        print("Path found. Generating trajectory")
        traj = self._generate_trajectory(path)


        print("Trajectory generated. Starting display")
        if display:
            self.display_trajectory(traj)

        return path, traj

    def display_trajectory(self, traj, meshcat=True, plotly=False):
        if meshcat:
            num_points = int((traj.end_time() - traj.start_time()) * 4000)
            for t in np.linspace(traj.start_time(), traj.end_time(), num_points):
                q = traj.value(t)
                # self.plant.GetJointByName("left_finger_sliding_joint", gripper).set_translation(self.plant_context, q[2])
                # # self.plant.GetJointByName("right_finger_sliding_joint", gripper).set_translation(self.plant_context, q[3])
                # self.plant.GetJointByName("base_revolute_joint", gripper).set_angle(self.plant_context, q[1])
                # self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[0])
                self.plant.GetJointByName("panda_finger_joint1", panda_hand).set_translation(self.plant_context, q[1][0])
                self.plant.GetJointByName("panda_joint7", panda_arm).set_angle(self.plant_context, q[0][0])
                self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[2][0])
                self.diagram.ForcedPublish(self.diagram_context)
                time.sleep(0.01)
        path = [traj.value(t) for t in np.linspace(traj.start_time(), traj.end_time(), 100)]
        path_q = [q.ravel() for q in path] # Convert to list of numpy arrays
        if plotly:
            self.visualize_cspace(factor=1, num_points=30, paths=[path_q], filled_polytopes=self.cs_free)
        return path_q

    def _generate_trajectory(self, path):
        trajs = []
        print("Generating trajectory from:")
        for q0, q1 in zip(path[:-1], path[1:]):
            print("From\t{}\tto\t{}".format(q0, q1))
            traj, result = self._generate_edge_trajectory(q0, q1)
            if not result.is_success():
                print("failed to generate edge trajectory from {} to {}".format(q0, q1))
                return None
            trajs.append(traj)
        return CompositeTrajectory.AlignAndConcatenate(trajs)


    def _generate_edge_trajectory(self, x_init, x_goal, path_length_weight=None):
        trajopt = GcsTrajectoryOptimization(self.plant.num_positions())
        gcs_regions = trajopt.AddRegions(self.cs_free, order=1, h_min=0.01)
        source = trajopt.AddRegions([Point(x_init)], order=0)
        target = trajopt.AddRegions([Point(x_goal)], order=0)
        trajopt.AddEdges(source, gcs_regions)
        trajopt.AddEdges(gcs_regions, target)
        trajopt.AddPathLengthCost(path_length_weight if path_length_weight is not None else 1.0)
        options = GraphOfConvexSetsOptions()
        [traj, result] = trajopt.SolvePath(source, target, options)
        print(f"result.is_success() = {result.is_success()}")
        print(f"result.get_solution_result() = {result.get_solution_result()}")
        print(f"result.get_solver_id().name() = {result.get_solver_id().name()}")


        # print("\n--- GCS RESULT ---")
        # print("success:", result.is_success())
        # print("solution result:", result.get_solution_result())
        # print("solver:", result.get_solver_id().name())

        # # try:
        # #     details = result.get_solver_details()
        # #     print("solver details:", details)

        # #     for field in [
        # #         "rescode",
        # #         "solution_status",
        # #         "optimizer_time",
        # #     ]:
        # #         if hasattr(details, field):
        # #             print(f"{field}:", getattr(details, field))
        # # except Exception as e:
        # #     print("Could not read solver details:", e)

        # print("------------------\n")

        return traj, result


    def _find_minimum_grasp_path(
            self,
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray,
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:

        for j in range(self.max_grasps):
            i = j + 1
            if i % 10 == 0 and i > 0:
                print(f"Trying with {i} grasps")
            path = self._solve_for_n_grasps_CC(
                i,
                x_init,
                x_goal,
                P,
                G,
                lower_joint_limits,
                upper_joint_limits,
                c_free_polytopes
            )

            if path is not None:
                print(f"Found a solution with {i} grasps")
                return path
        print(f"No solution found for less than {self.max_grasps} grasps")


    def _solve_for_n_grasps_CC(
            self,
            n_grasps: int,
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray,
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:
        # Initialize the program
        prog = MathematicalProgram()
        n_points = 2 * n_grasps + 2 # each grasp has a grasp and a release point, plus the initial and goal points
        n_vars = self.q_dim * n_points # for each point of q_dim length, we have n_points*q_dim variables
        non_gripper_dim = self.q_dim - self.gripper_dim # number of non-gripper variables per point

        # Create decision variables
        x = prog.NewContinuousVariables(n_vars, "x")

        # 1. Cost function: Minimize sum of squared distances between consecutive points
        """ ||x_{i+1} - x_{i}||^2  equiv || Cx - c ||^2  == 0 """

        c = np.zeros((n_vars - self.q_dim, 1))
        C = np.zeros((n_vars - self.q_dim, n_vars))
        C[:, self.q_dim:] = np.eye(n_vars - self.q_dim)
        C[:, :-self.q_dim] -= np.eye(n_vars - self.q_dim)

        Q, b = self._to_quadratic_form(C, c)
        Q += np.eye(Q.shape[1]) * 1e-6  # Add a small value to the diagonal to make Q positive definite

        prog.AddQuadraticCost(Q=Q, b=b, vars=x)

        # 2. Initial and goal constraints
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_init.flatten(), x[:self.q_dim])
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_goal.flatten(), x[-self.q_dim:])


        ###### Placement and Grasping Constraints should be adapted task specific #####

        # 3. Placement constraints (equality) -> Cap remains constant in transit paths
        """x_cap_{i+1} - x_cap_{i} == 0 for all even i, odd i+1"""
        for i in range(n_grasps + 1):
            idx = 2*self.q_dim*i # points to first variable of x_i for even i's
            idx1_cap = idx + self.q_dim - 1 # points to last variable of x_i (cap orientation)
            idx2_cap = idx1_cap + self.q_dim # points to last variable of x_{i+1} (cap orientation)
            prog.AddLinearEqualityConstraint(x[idx2_cap] - x[idx1_cap] == 0)

        # 4. Grasp constraints (equality between gripper and cap orientations)
        """x_cap_i - x_wrist_i - x_cap_{i+1} + x_wrist_{i+1} == 0 for odd i, even i+1"""
        for i in range(n_grasps):

            idxp1 = 2*self.q_dim*(i+1) # this idx points to first variable of x_i+1
            idx   = idxp1 - self.q_dim # this idx points to first variable of x_i

            idxp1_wrist = idxp1 + non_gripper_dim
            idxp1_cap = idxp1 + self.q_dim - 1
            idx_wrist = idx + non_gripper_dim
            idx_cap = idx + self.q_dim - 1 # equiv to idxp1 - 1

            prog.AddLinearEqualityConstraint(x[idx_cap] - x[idx_wrist] - x[idxp1_cap] + x[idxp1_wrist] == 0)

        ###### end of manual constraints


        # 5. Inequality Constraints (Placement hull)
        for i in range(n_points):
            prog.AddLinearConstraint(
                P.A(),  # Coefficient matrix
                -np.inf * np.ones_like(P.b()),  # Lower bound
                P.b(),  # Upper bound
                x[self.q_dim*i:self.q_dim*(i+1)]
            )
        # 6. Inequality Constraints (Grasp hull)
        # for i in range(n_points - 2):
        #     prog.AddLinearConstraint(
        #         G.A(),
        #         -np.inf * np.ones_like(G.b()),
        #         G.b(),
        #         x[self.q_dim*(i+1):self.q_dim*(i+2)]
        #     )

        # 7. Joint limits (inequality)
        prog.AddBoundingBoxConstraint(
            np.tile(lower_joint_limits, n_points),
            np.tile(upper_joint_limits, n_points),
            x
        )

        # 7. Collision-free polytope constraints (MIP)
        # Ensure all points are contained in at least one polytope

        M = 1e6  # Big-M constant (adjust based on problem scale)
        for i in range(n_points):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]

            # Creates one bineary 0-1 variable for each polytope
            z_i = prog.NewBinaryVariables(len(c_free_polytopes), name=f"z_{i}")

            # Each point x_i must be contained in at least one polytope
            prog.AddLinearConstraint(sum(z_i) >= 1)

            for j, poly in enumerate(c_free_polytopes):
                A_j = poly.A()
                b_j = poly.b()
                z_j = z_i[j]

                # Add constraints for each row of the polytope
                for k in range(A_j.shape[0]):
                    # Construct coefficient matrix [A_j_row | M]
                    coeffs = np.hstack([A_j[k], M])
                    # Combine x_i and z_j into variable vector
                    vars = np.hstack([x_i, [z_j]])
                    # A_j @ x_i + M * z[j] <= b_j + M
                    prog.AddLinearConstraint(
                        coeffs,
                        -np.inf,
                        b_j[k] + M,
                        vars
                    )

        # 8. Connected components constraints
        connected_components = self._compute_connected_components(P, G, c_free_polytopes)
        K = len(connected_components)

        # create binary variables for component membership (excluding init ang goal)
        z = {}
        for i in range (1, n_points-1): # exclude init and goal
            z[i] = prog.NewBinaryVariables(K, f"z_{i}")
            # Each point must belong to exactly one component
            prog.AddLinearConstraint(sum(z[i]) == 1)

        # Consecutive points must belong to the same component, ignoring init and goal
        for i in range(1, n_points-2, 2):
            for k in range(K):
                prog.AddLinearConstraint(z[i][k] == z[i+1][k])

        # Enforce that each component contains at least one point
        for i in range(1, n_points-1):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]
            z_i = z[i]

            for k, comp in enumerate(connected_components):
                A_k = comp.A()
                b_k = comp.b()
                z_k = z_i[k]

                for j in range(A_k.shape[0]):
                    coeffs = np.hstack([A_k[j], M])
                    vars = np.hstack([x_i, [z_k]])
                    prog.AddLinearConstraint(coeffs, -np.inf, b_k[j] + M, vars)

        # Solve the problem
        solver = MosekSolver()
        result = solver.Solve(prog)

        if not result.is_success():
            return None

        return result.GetSolution(x).reshape(-1, self.q_dim)

    @staticmethod
    def _to_quadratic_form(C, c):
        return np.dot(C.T, C), -np.dot(C.T, c).flatten()

    def _compute_connected_components(
        self,
        placement: HPolyhedron,
        grasp: HPolyhedron,
        c_free: list[HPolyhedron]
    ) -> list[HPolyhedron]:
        """
        Computes convex connected components as:
        {placement ∩ grasp ∩ (union of c_free)}.
        Returns convex hulls of connected regions.
        """
        components = []

        # Compute intersections with each c_free polytope
        for poly in c_free:
            intersection = placement.Intersection(grasp).Intersection(poly)
            if not intersection.IsEmpty():
                components.append(intersection)

        # Merge overlapping components
        merged = []
        for comp in components:
            add_to_merged = False
            for m in merged:
                if comp.Intersection(m).IsEmpty():
                    continue
                # Merge overlapping components
                vertices_comp = visualizer.get_polytope_vertices(comp)
                vertices_m = visualizer.get_polytope_vertices(m)
                vertices = np.vstack([vertices_comp, vertices_m])
                # QJ (joggle input): scipy's own recommendation for ConvexHull
                # precision failures in 5-D+ - near-degenerate/thin regions
                # (e.g. a very anisotropic grasp polytope) can otherwise raise
                # QhullError here.
                merged_hull = ConvexHull(vertices, qhull_options='QJ')
                
                vertices = np.vstack([vertices_comp, vertices_m])

                # Remove exactly/nearly duplicated vertices
                vertices = np.unique(np.round(vertices, decimals=12), axis=0)

                centered = vertices - vertices.mean(axis=0)

                s = np.linalg.svd(centered, compute_uv=False)

                print("Number of vertices:", len(vertices))
                print("Affine rank:", np.linalg.matrix_rank(centered))
                print("Singular values:")
                print(s)
                print("relative singular values:")
                print(s / s[0])


                merged_poly = convex_hull_to_hpolyhedron(merged_hull)
                merged.remove(m)
                merged.append(merged_poly)
                add_to_merged = True
                break
            if not add_to_merged:
                merged.append(comp)

        return merged



    def _generate_cfree(self, irisalg: str = "irisnp") -> list[HPolyhedron]:
        if irisalg not in ["irisnp", "irisnp2", "iriszo"]:
            raise ValueError(f"Invalid irisalg: {irisalg}. Must be one of ['irisnp', 'irisnp2', 'iriszo']")

        generator = RandomGenerator(1234)
        checker = SceneGraphCollisionChecker(
            model=self.diagram,
            robot_model_instances=self.model_instances,
            edge_step_size=0.01,
        )


        if irisalg != "irisnp":
            # For iriszo and irisnp2, we need to set the sampled iris options
            common_sampled_iris_options = CommonSampledIrisOptions()
            common_sampled_iris_options.delta = 0.001
            common_sampled_iris_options.epsilon = 0.0001
            common_sampled_iris_options.max_iterations = 50
            common_sampled_iris_options.max_iterations_separating_planes = 50
            # common_sampled_iris_options.mixing_steps =
            common_sampled_iris_options.num_particles = 500
            common_sampled_iris_options.parallelism = Parallelism(32)
            common_sampled_iris_options.verbose = True
            common_sampled_iris_options.configuration_space_margin = 1e-4
            common_sampled_iris_options.termination_threshold = 1e-5
            common_sampled_iris_options.relative_termination_threshold = 1e-4
            common_sampled_iris_options.remove_all_collisions_possible = True

        options = IrisFromCliqueCoverOptions()
        options.num_points_per_visibility_round = 200
        options.coverage_termination_threshold = 0.99
        options.parallelism = Parallelism(32)

        if irisalg == "iriszo":
            iris_zo_options = IrisZoOptions()
            iris_zo_options.bisection_steps = 50
            iris_zo_options.sampled_iris_options = common_sampled_iris_options
            options.iris_options = iris_zo_options
        elif irisalg == "irisnp2":
            iris_np2_options = IrisNp2Options()
            iris_np2_options.sampled_iris_options = common_sampled_iris_options
            options.iris_options = iris_np2_options
        else:
            # irisnp uses the default IrisOptions object on options.iris_options.
            options.iris_options.configuration_space_margin = 1e-4

        # See https://github.com/RobotLocomotion/drake/issues/21343
        regions = IrisInConfigurationSpaceFromCliqueCover(checker, options, generator, [])
        print("\nnumber of regions: ", len(regions))
        return regions

## 4. Generate C-free (native Drake IRIS-ZO) and plan a trajectory

**Caveats, read before running:**
- `cs_free=None, irisalg="iriszo"` makes `__init__` call the class's own
  `_generate_cfree(irisalg="iriszo")` - clique-cover to 98% coverage over
  the full 10-dim configuration space, `Parallelism(32)`. This is the
  expensive step (minutes+, depends on hardware) and has not been run as
  part of writing this notebook.

In [16]:
# Optionally load a precomputed set of collision-free polytopes from a file
from pydrake.geometry.optimization import LoadIrisRegionsYamlFile

cfree_regions_dict = LoadIrisRegionsYamlFile("data/cfree/cfree_10dof_irisnp2_98_k42.yaml")
cfree_regions = list(cfree_regions_dict.values())


In [17]:
model_instances = [panda_arm, panda_hand, cap, obstacle1, obstacle3]

planner = ManipulationPlanner(
    visualizer=visualizer,
    CP=placement_polytope,
    CG=grasp_polytope,
    max_grasps=20,
    gripper_dim=3,
    model_instances=model_instances,
    cs_free=cfree_regions,
    irisalg="irisnp2"
)

In [18]:
from pydrake.geometry.optimization import SaveIrisRegionsYamlFile

# SaveIrisRegionsYamlFile(
#     "data/cfree/cfree_10dof_irisnp2_99_k52.yaml",
#     {
#         f"region_{i:03d}": region
#         for i, region in enumerate(planner.cs_free)
#     },
# )

# np.save(
#     "path_drake_1_49.npy",
#     path,
# )

## Fixing free polytopes
Some of the free polytopes generated by either iris-zo and iris-np2 collide with the obstacles in some of the polytopes. Here, we'll try to go free polytope by free polytope that intersects with the grasp polytope. In the intersection with the grasp polytope, we interpolate along the wrist axis and find collisions with the collision checker. We then add hyperplanes to the free polytope accordingly to remove collisions, in perpendicular direction to the wrist axis.

1. **Wrist sweep.** Same sweep as the brute force below: `grasp_configuration` with the wrist from −1 to 1, each sample checked with the collision checker.
2. **Free segment.** For every free polytope F, the collision-free samples inside it form its free wrist segment. We keep the longest one, as the brute force does.
3. **False positives.** A sample the collision checker says collides but some F says is free. For every false positive q_c and every F that contains it, add one plane to F:
   - NLP (only constraint: don't collide): q_f = argmin ‖q − q_c‖², seeded at q_c, cap fixed. q_f is on the border of the obstacle and q_c − q_f is the direction that gets out of collision, so the plane (q_c − q_f)ᵀ q ≤ (q_c − q_f)ᵀ q_f goes through q_f, is tangent to the obstacle there and cuts q_c off.
   - If the NLP fails or its plane would also cut the free segment of F, use a plane perpendicular to the wrist axis at the end of the free segment instead.
4. **Check.** Count the false positives again and replace `planner.cs_free` with the fixed polytopes, so the brute force and the A* below use them.

**Notes.**
- The fingers must stay at ±0.0245 as in the brute force. At q0's ±0.02405 the finger spheres already touch the cap (~0.1 mm), which the collision checker counts as a collision at every wrist angle.
- The cap angle does not change the collisions (cap 0.94 and 3.0 give the same result), so dropping the cap in the A* is fine.
- The NLP gets out of collision mostly by moving the other arm joints by ~0.01 rad (the obstacle penetration is only 4-6 mm), not the wrist. Its plane is then almost parallel to the wrist axis and would also cut the free segment, so in practice the planes added are the wrist ones.

In [19]:
NLP_MARGIN = 1e-3    # distance the NLP asks for, a bit more than the collision checker's 1e-5

e_wrist = np.zeros(plant.num_positions())
e_wrist[idx_wrist] = 1.0


def longest_free_run(sweep_in_F):
    """Longest run of consecutive collision-free samples: the free wrist segment of a polytope (as in the brute force)."""
    best, run = [], []
    for q, free in sweep_in_F:
        run = run + [q] if free else []
        if len(run) > len(best):
            best = run
    return best


def separating_plane(q_c, keep):
    """Plane n @ q <= d that cuts the colliding configuration q_c off and keeps the samples in keep.

    NLP (only constraint: don't collide): q_f = argmin ||q - q_c||^2, seeded at q_c, cap fixed.
    q_f is on the border of the obstacle and n = q_c - q_f is the direction that gets out of
    collision, so the plane through q_f with normal n is tangent to the obstacle there.
    Fallback (NLP fails, or its plane would also cut keep): plane perpendicular to the wrist axis
    at the end of the free segment.
    """
    ik = InverseKinematics(plant, plant_context)
    ik.AddMinimumDistanceLowerBoundConstraint(NLP_MARGIN)
    q = ik.q()
    prog = ik.get_mutable_prog()
    prog.AddBoundingBoxConstraint(q_c[idx_cap], q_c[idx_cap], q[idx_cap])   # no cap component in the plane
    prog.AddQuadraticErrorCost(np.eye(len(q_c)), q_c, q)
    result = Solve(prog, q_c)
    if result.is_success():
        q_f = result.GetSolution(q)
        n = q_c - q_f
        if np.linalg.norm(n) > 1e-9:
            n = n / np.linalg.norm(n)
            d = n @ q_f
            if all(n @ q_k <= d + 1e-9 for q_k in keep):
                return n, d, "nlp"
    if not keep:
        return None
    w_keep = [q_k[idx_wrist] for q_k in keep]
    if q_c[idx_wrist] > max(w_keep):
        return e_wrist, max(w_keep), "wrist"      # wrist <= end of the free segment
    return -e_wrist, -min(w_keep), "wrist"        # wrist >= start of the free segment

In [20]:
# Same wrist sweep as the brute force (next section)
grasp_configuration = find_grasp_seed(grasp_constraints)
grasp_configuration[idx_f1] = -0.0245
grasp_configuration[idx_f2] = 0.0245
grasp_configuration[idx_cap] = 3.0
wrist_angles = np.linspace(-1., 1., num=100)

sweep = []    # (q, collision-free according to the collision checker)
for w in wrist_angles:
    q = grasp_configuration.copy()
    q[idx_wrist] = w
    sweep.append((q, not visualizer.check_collision_q_by_ik(q)))

if not hasattr(planner, "cs_free_iris"):   # keep the IRIS polytopes, so this cell can be re-run
    planner.cs_free_iris = list(planner.cs_free)
cfree_fixed = list(planner.cs_free_iris)

# free wrist segment of every polytope: longest run of free samples inside it
keep = [longest_free_run([(q, free) for q, free in sweep if F.PointInSet(q)]) for F in cfree_fixed]

planes = {}   # k -> kinds of the planes added to F_k
for q, free in sweep:
    if free:
        continue
    for k, F in enumerate(cfree_fixed):
        if not F.PointInSet(q):
            continue
        # false positive: the collision checker says collision, F_k says free
        plane = separating_plane(q, keep[k])
        if plane is None:
            print(f"F_{k}: no plane for the false positive at wrist {q[idx_wrist]:+.3f}")
            continue
        n, d, kind = plane
        cfree_fixed[k] = HPolyhedron(np.vstack([F.A(), n]), np.append(F.b(), d))
        planes.setdefault(k, []).append(kind)

for k, kinds in planes.items():
    seg = f"[{keep[k][0][idx_wrist]:+.3f}, {keep[k][-1][idx_wrist]:+.3f}]" if keep[k] else "none"
    print(f"F_{k:<3d} free segment {seg}  planes: {kinds.count('nlp')} nlp, {kinds.count('wrist')} wrist")

F_6   free segment [-1.000, -0.192]  planes: 0 nlp, 1 wrist
F_7   free segment [-1.000, -0.192]  planes: 0 nlp, 1 wrist
F_38  free segment [-1.000, -0.192]  planes: 0 nlp, 1 wrist
F_40  free segment [-1.000, -0.192]  planes: 0 nlp, 1 wrist
F_0   free segment [+0.919, +0.919]  planes: 0 nlp, 1 wrist


In [21]:
def count_false_positives(polytopes):
    return sum(not free and any(F.PointInSet(q) for F in polytopes) for q, free in sweep)


print(f"false positives: {count_false_positives(planner.cs_free_iris)} before, "
      f"{count_false_positives(cfree_fixed)} after (out of {len(sweep)} wrist angles)")

planner.cs_free = cfree_fixed   # the brute force and the A* below now use the fixed polytopes

# SaveIrisRegionsYamlFile(
#     "data/cfree/cfree_10dof_irisnp2_98_k42_fixed.yaml",
#     {f"region_{i:03d}": region for i, region in enumerate(planner.cs_free)},
# )

false positives: 47 before, 0 after (out of 100 wrist angles)


## Brute Force trial

Try to reach grasp polytope from q_init, then turn as much as possible so that the cap reaches their goal angle, then take the arm to q_goal without moving the cap. 

1. Find a grasp configuration
2. Turn the wrist and draw limits based on wrist rotation (that's one hyperplane inequality). 
3. Find largest wrist rotation segment that is feasible (i.e. does not collide with obstacles) based on the limits from step 2.
4. Move the arm to the the start configuration of the larger wrist rotation segment
5. Move the wrist to the end configuration of the larger wrist rotation segment, moving the cap in the same measure as the wrist rotation (i.e. transfer).
6. Return the wrist to the start configuration of the larger wrist rotation segment, with the cap still (i.e. transit)
7. Repeat 5-6 until the cap reaches the goal angle (last element of the q_goal vector).
8. Move the arm to the q_goal configuration, with the cap still (i.e. transit).



In [22]:
grasp_configuration = find_grasp_seed(grasp_constraints)
grasp_configuration[idx_f1] = -0.0245
grasp_configuration[idx_f2] = 0.0245
grasp_configuration[idx_cap] = 3.0
plant.SetPositions(plant_context, grasp_configuration)
diagram.ForcedPublish(diagram_context)
print("ciris_plant_visualizer collision check:", visualizer.check_collision_q_by_ik(grasp_configuration))
# check if the grasp configuration is in the grasp polytope
print("Grasp configuration in grasp polytope:", grasp_polytope.PointInSet(grasp_configuration))
print("Grasp configuration in any c-free polytope:", any(poly.PointInSet(grasp_configuration) for poly in planner.cs_free))

# do linespace between wrist angles from grasp_configuration, and check for collisions along the way to identify collision-free segments
wrist_angles = np.linspace(-1., 1., num=100)
collision_free_segments = []
current_segment = []
cfree_polytpes = planner.cs_free

for i in range(len(wrist_angles) - 1):
    q = grasp_configuration.copy()
    q[idx_wrist] = wrist_angles[i]
    q_cap = q.copy()
    q_cap[idx_cap] = grasp_configuration[idx_cap] - (wrist_angles[i] - wrist_angles[0])
    q_next = grasp_configuration.copy()
    q_next[idx_wrist] = wrist_angles[i + 1]
    plant.SetPositions(plant_context, q_cap)
    diagram.ForcedPublish(diagram_context)
    time.sleep(0.1)

    # check if both configurations are in the grasp polytope\
    if grasp_polytope.PointInSet(q) and grasp_polytope.PointInSet(q_next):
        # check if both configurations are in any of the c-free polytopes
        in_cfree_q = any(poly.PointInSet(q) for poly in cfree_polytpes)
        in_cfree_q_next = any(poly.PointInSet(q_next) for poly in cfree_polytpes)
        # in_cfree_q = not visualizer.check_collision_q_by_ik(q)
        # in_cfree_q_next = not visualizer.check_collision_q_by_ik(q_next)
        
        print(f"q: {q[idx_wrist]:.3f}, q_next: {q_next[idx_wrist]:.3f}, in_cfree_q: {in_cfree_q}, in_cfree_q_next: {in_cfree_q_next}, q_cap: {q_cap[idx_cap]:.3f}")

        if in_cfree_q and in_cfree_q_next:
            current_segment.append((q, q_next))
        else:
            print(f"Collision detected at wrist angles: {q[idx_wrist]:.3f} or {q_next[idx_wrist]:.3f}")
            # input("Press Enter to continue...")
            if current_segment:
                collision_free_segments.append(current_segment)
                current_segment = []
    else:
        print(f"Configuration not in grasp polytope at wrist angles: {q[idx_wrist]:.3f} or {q_next[idx_wrist]:.3f}")
            
            
print(f"Found {len(collision_free_segments)} collision-free segments along the wrist angle sweep.")


ciris_plant_visualizer collision check: 1.0
Grasp configuration in grasp polytope: True
Grasp configuration in any c-free polytope: False
q: -1.000, q_next: -0.980, in_cfree_q: True, in_cfree_q_next: True, q_cap: 3.000
q: -0.980, q_next: -0.960, in_cfree_q: True, in_cfree_q_next: True, q_cap: 2.980
q: -0.960, q_next: -0.939, in_cfree_q: True, in_cfree_q_next: True, q_cap: 2.960
q: -0.939, q_next: -0.919, in_cfree_q: True, in_cfree_q_next: True, q_cap: 2.939
q: -0.919, q_next: -0.899, in_cfree_q: True, in_cfree_q_next: True, q_cap: 2.919
q: -0.899, q_next: -0.879, in_cfree_q: True, in_cfree_q_next: True, q_cap: 2.899
q: -0.879, q_next: -0.859, in_cfree_q: True, in_cfree_q_next: True, q_cap: 2.879
q: -0.859, q_next: -0.838, in_cfree_q: True, in_cfree_q_next: True, q_cap: 2.859
q: -0.838, q_next: -0.818, in_cfree_q: True, in_cfree_q_next: True, q_cap: 2.838
q: -0.818, q_next: -0.798, in_cfree_q: True, in_cfree_q_next: True, q_cap: 2.818
q: -0.798, q_next: -0.778, in_cfree_q: True, in_cfre

### Get number of FP, TP, FN and TN with the collision checker and the c-free polytopes.

In [23]:

print("Collision-free segments:")
for i, segment in enumerate(collision_free_segments):
    print(f"Segment {i}: {len(segment)} pairs of configurations")
    for q, q_next in segment:
        print(f"  q: {q[idx_wrist]:.3f}, q_next: {q_next[idx_wrist]:.3f}")

Collision-free segments:
Segment 0: 40 pairs of configurations
  q: -1.000, q_next: -0.980
  q: -0.980, q_next: -0.960
  q: -0.960, q_next: -0.939
  q: -0.939, q_next: -0.919
  q: -0.919, q_next: -0.899
  q: -0.899, q_next: -0.879
  q: -0.879, q_next: -0.859
  q: -0.859, q_next: -0.838
  q: -0.838, q_next: -0.818
  q: -0.818, q_next: -0.798
  q: -0.798, q_next: -0.778
  q: -0.778, q_next: -0.758
  q: -0.758, q_next: -0.737
  q: -0.737, q_next: -0.717
  q: -0.717, q_next: -0.697
  q: -0.697, q_next: -0.677
  q: -0.677, q_next: -0.657
  q: -0.657, q_next: -0.636
  q: -0.636, q_next: -0.616
  q: -0.616, q_next: -0.596
  q: -0.596, q_next: -0.576
  q: -0.576, q_next: -0.556
  q: -0.556, q_next: -0.535
  q: -0.535, q_next: -0.515
  q: -0.515, q_next: -0.495
  q: -0.495, q_next: -0.475
  q: -0.475, q_next: -0.455
  q: -0.455, q_next: -0.434
  q: -0.434, q_next: -0.414
  q: -0.414, q_next: -0.394
  q: -0.394, q_next: -0.374
  q: -0.374, q_next: -0.354
  q: -0.354, q_next: -0.333
  q: -0.333, 

### Find the largest segment of wrist rotation that is feasible (i.e. does not collide with obstacles) based on the limits from step 2. 

Then 
- find a free polytope where the lower wrist rotation limit is.
- find a free polytope where the upper wrist rotation limit is.
- Set init and goal configurations on those polytopes respectively. (the farther away from the grasp configuration as possible)
- Go from init configuration to grasping configuration at the lower wrist rotation limit (i.e. transit).
- Get the difference from the init cap configuration to the goal cap configuration, and divide it by the difference from the lower wrist rotation limit to the upper wrist rotation limit. the module will give us the number of grasping cycles we need to do to reach the goal cap configuration.
- Then: if the number of grasping cycles is greater than 1, we will repeat the following steps until the cap reaches the goal angle (last element of the q_goal vector):
    - Move the wrist to the end configuration of the larger wrist rotation segment, moving the cap in the same measure as the wrist rotation (i.e. transfer).
    - Return the wrist to the start configuration of the larger wrist rotation segment, with the cap still (i.e. transit)

In [24]:
# find the largest collision-free segment
largest_segment = max(collision_free_segments, key=len)
print(f"Largest collision-free segment has {len(largest_segment)} pairs of configurations")

# Get the first and last configurations of the largest segment
lower_segment_limit = largest_segment[0][0]
upper_segment_limit = largest_segment[-1][1]
print(f"Lower segment limit: {lower_segment_limit[idx_wrist]:.3f}, Upper segment limit: {upper_segment_limit[idx_wrist]:.3f}")

init_cap_angle = lower_segment_limit[idx_cap]
goal_cap_angle = -3.0
# upper_segment_limit[idx_cap] = goal_cap_angle
print(f"Initial cap angle: {init_cap_angle:.3f}, Goal cap angle: {goal_cap_angle:.3f}")

#set goal cap angle to 3.


# Find a free polytope that contains the lower and upper segment limits
lower_poly = None
upper_poly = None
for poly in cfree_polytpes:
    if poly.PointInSet(lower_segment_limit):
        lower_poly = poly
    if poly.PointInSet(upper_segment_limit):
        upper_poly = poly
        
if lower_poly is not None and upper_poly is not None:
    print("Found polytopes containing the lower and upper segment limits.")
    
    
# The initial and goal configurations for the manipulation planner will be the centroids of the lower and upper polytopes
lower_poly_centroid = np.mean(visualizer.get_polytope_vertices(lower_poly), axis=0)
upper_poly_centroid = np.mean(visualizer.get_polytope_vertices(upper_poly), axis=0)



def lerp(start, end, num_points=1000):
    return np.array([start + (end - start) * t for t in np.linspace(0, 1, num_points)])


Largest collision-free segment has 40 pairs of configurations
Lower segment limit: -1.000, Upper segment limit: -0.192
Initial cap angle: 3.000, Goal cap angle: -3.000
Found polytopes containing the lower and upper segment limits.


In [25]:
cap_config = lower_poly_centroid.copy()
cap_config[idx_cap] = init_cap_angle
# cap_config[idx_cap] = goal_cap_angle
print(f"Setting cap angle to goal: {cap_config[idx_cap]:.3f}")
plant.SetPositions(plant_context, cap_config)
diagram.ForcedPublish(diagram_context)


Setting cap angle to goal: 3.000


## Reachability A* over free and grasp polytopes

Inspired by CASSR (Wang & Tonneau, arXiv 2603.02989): an A* **node is a polytope plus the subset of it
reachable from the start**, not a single configuration. Expanding a node propagates reachability into
neighbouring polytopes.

In our problem the state is (arm configuration q, cap angle α):
- **Transit** in a free region F_i: the arm can reach any point of F_i (it is convex), and α is frozen.
- **Grasp** in G = Grasp ∩ F: the wrist is the only real degree of freedom. A forward stroke (wrist ↑)
  turns the cap, α −= Δw. A return stroke (wrist ↓) keeps α (ratchet).

So a node's reachable set is always *(the whole polytope) × (an interval of α)*, and propagation is just
interval arithmetic. A* minimises the number of grasps (forward strokes), ties broken by fewer polytope
hops. The actual configurations are placed afterwards, on the chosen polytope sequence.


### Step 1 — Drop the cap from every polytope
We ignore the cap in the polytopes: its column is deleted from every C-free region and from the grasp
polytope (rows that only bounded the cap, i.e. its joint limits, disappear). The cap angle α is taken
from q_init / q_goal and only changes through the grasp pieces (steps 5-6).

**Trust region on the grasp polytope.** `grasp_polytope` linearises h(q) around q0, but Jh only has rank 4,
so the linear constraints leave a 6-D null space free: the arm could slide ~2 rad away from q0 (gripper
0.8 m from the cap, |h| ≈ 0.95) and still be "inside". The linearisation is only valid near q0, so we keep
the 6 non-wrist arm joints within ±`GRASP_TRUST` of q0 and leave the wrist free: its axis passes through
the EE origin, so rotating it keeps the grasp (|h| stays ≈ 0.002 along the whole plan).

In [26]:
import itertools
from pydrake.solvers import MathematicalProgram, Solve

cap_lower = plant.GetPositionLowerLimits()[idx_cap]
cap_upper = plant.GetPositionUpperLimits()[idx_cap]
W = idx_wrist if idx_wrist < idx_cap else idx_wrist - 1   # wrist index in the 9-D (cap-free) space


def to_arm(q):
    """10-D configuration -> 9-D (cap removed)."""
    return np.delete(np.asarray(q, dtype=float), idx_cap)


def to_full(x, alpha):
    """9-D configuration + cap angle alpha -> 10-D configuration."""
    return np.insert(np.asarray(x, dtype=float), idx_cap, alpha)


def drop_cap(poly, tol=1e-9):
    """Remove the cap coordinate: the cap is ignored in the polytopes and tracked separately (α)."""
    A_arm = np.delete(poly.A(), idx_cap, axis=1)
    keep = np.linalg.norm(A_arm, axis=1) > tol    # rows that only bounded the cap (its joint limits) vanish
    return HPolyhedron(A_arm[keep], poly.b()[keep])


free_sets = [drop_cap(F) for F in planner.cs_free]
grasp9 = drop_cap(grasp_polytope)

# Trust region: the linearisation of h(q) is only valid near q0 -> keep the non-wrist arm joints near q0.
GRASP_TRUST = 0.05   # rad
x0 = to_arm(q0)
trust_lo, trust_hi = np.full(len(x0), -np.inf), np.full(len(x0), np.inf)
for j in range(7):                                   # the 7 arm joints (0..6 in the plant ordering)
    if j == idx_wrist:
        continue
    jj = j if j < idx_cap else j - 1
    trust_lo[jj], trust_hi[jj] = x0[jj] - GRASP_TRUST, x0[jj] + GRASP_TRUST
A_trust = np.vstack([np.eye(len(x0)), -np.eye(len(x0))])
b_trust = np.concatenate([trust_hi, -trust_lo])
finite = np.isfinite(b_trust)
grasp9 = grasp9.Intersection(HPolyhedron(A_trust[finite], b_trust[finite]))
print(f"grasp polytope: trust region ±{GRASP_TRUST} rad on the non-wrist arm joints")
print(f"{len(free_sets)} free regions, {sum(F.IsEmpty() for F in free_sets)} empty after dropping the cap")
print("q0 (arm) in grasp9:", grasp9.PointInSet(to_arm(q0)))

grasp polytope: trust region ±0.05 rad on the non-wrist arm joints
42 free regions, 0 empty after dropping the cap
q0 (arm) in grasp9: True


### Step 2 — Grasp pieces G_k = Grasp ∩ F_k
These are the switch sets: the only places where the cap can be turned. Each piece is essentially an
interval of wrist values; two small LPs (no binaries, no big-M) find its lowest- and highest-wrist points.

In [27]:
def extreme_point(poly, direction):
    """Point of poly minimising direction @ x (a small LP, no binaries)."""
    prog = MathematicalProgram()
    x = prog.NewContinuousVariables(poly.ambient_dimension(), "x")
    prog.AddLinearConstraint(poly.A(), np.full(len(poly.b()), -np.inf), poly.b(), x)
    prog.AddLinearCost(direction, 0.0, x)
    result = Solve(prog)
    if not result.is_success():
        raise RuntimeError("extreme_point LP failed")
    return result.GetSolution(x)


e_w = np.zeros(grasp9.ambient_dimension())
e_w[W] = 1.0

pieces = {}   # k -> piece of the grasp polytope inside free region k
for k, F in enumerate(free_sets):
    G = grasp9.Intersection(F)
    if G.IsEmpty():
        continue
    lo, hi = extreme_point(G, e_w), extreme_point(G, -e_w)
    pieces[k] = dict(poly=G, lo=lo, hi=hi, w_lo=lo[W], w_hi=hi[W])

print(f"{len(pieces)} grasp pieces:")
for k, p in sorted(pieces.items(), key=lambda kv: kv[1]["w_lo"]):
    print(f"  G_{k:<3d} wrist [{p['w_lo']:+.3f}, {p['w_hi']:+.3f}]  length {p['w_hi'] - p['w_lo']:.3f}")

8 grasp pieces:
  G_2   wrist [-1.000, -0.640]  length 0.360
  G_4   wrist [-1.000, -0.944]  length 0.056
  G_6   wrist [-1.000, -0.192]  length 0.808
  G_7   wrist [-1.000, -0.192]  length 0.808
  G_25  wrist [-1.000, -0.825]  length 0.175
  G_38  wrist [-1.000, -0.192]  length 0.808
  G_40  wrist [-1.000, -0.192]  length 0.808
  G_0   wrist [+0.715, +0.919]  length 0.204


### Step 3 — Grasp components (wrist segments)
Pieces that intersect form one connected wrist segment of length L_c: this is what the brute force
found by sweeping, but derived from the polytopes. A forward stroke can turn the cap by at most L_c.
The chain is a wrist-ordered list of points, each consecutive pair inside one piece (so lerps are safe).

**`MERGE_PIECES = False` (for now):** every straight-line move in the final path must lie inside *one*
convex set, which is what certifies it collision-free. Inside a single piece that is automatic. Across
merged pieces, the entry piece may not be the one the stroke chain runs through, and that move would not
be covered by any single polytope. So each piece is its own grasp component for now. With the current
regions every piece already spans the whole wrist range, so nothing is lost.

In [28]:
L_MIN = 0.05          # ignore segments too short to be useful for turning
MERGE_PIECES = False  # True: merge overlapping pieces into one segment (step 6 needs False)

piece_keys = sorted(pieces)
overlaps = {}
root = {k: k for k in piece_keys}


def find(k):
    while root[k] != k:
        root[k] = root[root[k]]
        k = root[k]
    return k


for a, b in itertools.combinations(piece_keys, 2):
    I = pieces[a]["poly"].Intersection(pieces[b]["poly"])
    if not I.IsEmpty():
        overlaps[(a, b)] = overlaps[(b, a)] = I
        root[find(a)] = find(b)

groups = {}
for k in piece_keys:
    groups.setdefault(find(k) if MERGE_PIECES else k, []).append(k)


def build_chain(member_keys):
    cur = min(member_keys, key=lambda k: pieces[k]["w_lo"])
    points, seg_piece = [pieces[cur]["lo"]], []
    while True:
        ahead = [k for k in member_keys
                 if (cur, k) in overlaps and pieces[k]["w_hi"] > pieces[cur]["w_hi"] + 1e-9]
        if not ahead:
            break
        nxt = max(ahead, key=lambda k: pieces[k]["w_hi"])
        points.append(extreme_point(overlaps[(cur, nxt)], -e_w))   # highest-wrist point of the overlap
        seg_piece.append(cur)
        cur = nxt
    points.append(pieces[cur]["hi"])
    seg_piece.append(cur)
    wrists = [p[W] for p in points]
    if np.any(np.diff(wrists) < -1e-9):
        print(f"  warning: chain not monotone in wrist: {np.round(wrists, 3)}")
    return points, seg_piece


components = []
for member_keys in groups.values():
    chain, seg_piece = build_chain(member_keys)
    L = chain[-1][W] - chain[0][W]
    if L < L_MIN:
        continue
    components.append(dict(pieces=member_keys, chain=chain, seg_piece=seg_piece,
                           w_min=chain[0][W], w_max=chain[-1][W], L=L))

L_MAX = max(c["L"] for c in components)
print(f"{len(components)} grasp components (segments), longest L = {L_MAX:.3f}")
for c_id, c in enumerate(components):
    print(f"  C{c_id}: wrist [{c['w_min']:+.3f}, {c['w_max']:+.3f}]  L={c['L']:.3f}  pieces {sorted(c['pieces'])}")

if "collision_free_segments" in globals():
    print("brute-force segments (collision checker):")
    for seg in collision_free_segments:
        print(f"  wrist [{seg[0][0][idx_wrist]:+.3f}, {seg[-1][1][idx_wrist]:+.3f}]")

8 grasp components (segments), longest L = 0.808
  C0: wrist [+0.715, +0.919]  L=0.204  pieces [0]
  C1: wrist [-1.000, -0.640]  L=0.360  pieces [2]
  C2: wrist [-1.000, -0.944]  L=0.056  pieces [4]
  C3: wrist [-1.000, -0.192]  L=0.808  pieces [6]
  C4: wrist [-1.000, -0.192]  L=0.808  pieces [7]
  C5: wrist [-1.000, -0.825]  L=0.175  pieces [25]
  C6: wrist [-1.000, -0.192]  L=0.808  pieces [38]
  C7: wrist [-1.000, -0.192]  L=0.808  pieces [40]
brute-force segments (collision checker):
  wrist [-1.000, -0.192]


### Step 4 — Polytope graph
Edges connect sets with a nonempty intersection:
- F_i ↔ F_j (transit): door = Chebyshev centre of F_i ∩ F_j.
- F_i ↔ component c (switch): through a piece G_k with F_i ∩ G_k ≠ ∅. We store the highest wrist value
  there (best entry) and the lowest (best exit): moving the wrist forward while grasping always turns
  the cap, so exiting "ahead" of the entry forces some turning.

Computed eagerly here for clarity (CASSR builds this lazily, which is a later optimisation).

In [29]:
n_free = len(free_sets)

ff_door = {}      # (i, j) -> Chebyshev centre of F_i ∩ F_j
for i, j in itertools.combinations(range(n_free), 2):
    I = free_sets[i].Intersection(free_sets[j])
    if not I.IsEmpty():
        ff_door[(i, j)] = ff_door[(j, i)] = I.ChebyshevCenter()
free_neighbors = {i: [j for j in range(n_free) if (i, j) in ff_door] for i in range(n_free)}

fc_links = {}     # (i, c) -> list of dicts: piece k, entry/exit points and wrist values
for i in range(n_free):
    for c_id, comp in enumerate(components):
        links = []
        for k in comp["pieces"]:
            I = free_sets[i].Intersection(pieces[k]["poly"])
            if I.IsEmpty():
                continue
            entry, exit_ = extreme_point(I, -e_w), extreme_point(I, e_w)
            links.append(dict(piece=k, entry=entry, entry_w=entry[W], exit=exit_, exit_w=exit_[W]))
        if links:
            fc_links[(i, c_id)] = links
comps_near = {i: [c for c in range(len(components)) if (i, c) in fc_links] for i in range(n_free)}

print(f"transit edges F-F: {len(ff_door) // 2}")
print(f"switch edges F-C:  {len(fc_links)}")
for c_id in range(len(components)):
    print(f"  C{c_id} reachable from free regions {[i for i in range(n_free) if (i, c_id) in fc_links]}")

transit edges F-F: 850
switch edges F-C:  50
  C0 reachable from free regions [0]
  C1 reachable from free regions [2, 4, 6, 7, 25, 38, 40]
  C2 reachable from free regions [2, 4, 6, 7, 25, 38, 40]
  C3 reachable from free regions [2, 4, 6, 7, 25, 38, 40]
  C4 reachable from free regions [2, 4, 6, 7, 25, 38, 40]
  C5 reachable from free regions [2, 4, 6, 7, 25, 38, 40]
  C6 reachable from free regions [2, 4, 6, 7, 25, 38, 40]
  C7 reachable from free regions [2, 4, 6, 7, 25, 38, 40]


### Step 5 — A* search over (polytope, α-interval) nodes

**Node** = a polytope plus the set reachable in it: `P × [α_lo, α_hi]`.
- `kind="F"`: transit inside free region F_i. Any arm configuration of F_i is reachable (convex), α is frozen.
- `kind="C"`: inside grasp component c. Also remembers the wrist range where this visit entered
  (`entry_w_lo`, `entry_w`) and its **phase**:
  - `entered` — just arrived, nothing grasped yet (the wrist can be anywhere in the entry range);
  - `holding` — just finished a forward stroke, still grasping;
  - `released` — after a return (ungrasped, wrist rotated back), ready for a full stroke.

**Moves**, with the α-interval rule and cost of each. Cost = **grasp actions + ungrasp actions**, plus a
small price μ = `HOP_COST` per polytope change (approach motions, no contact):

| move | from → to | α interval | cost |
|---|---|---|---|
| transit | F_i → F_j (F_i ∩ F_j ≠ ∅) | unchanged | μ |
| enter   | F_i → C (F_i ∩ G ≠ ∅) | unchanged | μ |
| stroke (grasp) | `entered` → `holding` | `α_lo −= w_max − e_lo` (starts inside the entry range) | 1 |
| stroke (grasp) | `released` → `holding` | `α_lo −= L` | 1 |
| return (ungrasp) | `entered`/`holding` → `released` | unchanged | 1 |
| exit (release) | `holding` → F_j | `α_hi −= d` | 1 |
| exit (never grasped) | `entered` → F_j, only if d ≤ 0 | unchanged | μ |

d = max(0, lowest exit wrist − highest entry wrist): the forward wrist travel (= cap turning) that
reaching the exit forces. α_lo is clipped at α_goal.

Exit must cost 1: otherwise "exit and re-enter elsewhere" would be a free substitute for a return.
Note that a return *before* the first stroke is always avoidable with one extra hop: entering G_k from
F_i needs F_i ∩ G_k ≠ ∅, so F_i touches F_k, and F_k ∩ G_k = G_k covers the whole wrist range of the
piece. With μ < 1 the search therefore prefers that hop.

**Heuristic** h = 2·⌈(α_lo − α_goal)/L_max⌉ (+1 if `holding`): every remaining stroke must be followed by
an ungrasp (a return or the final release), and a holding node still has to release. Admissible, and
consistent (no move lowers h by more than its cost) → A* returns the fewest grasp + ungrasp actions,
ties broken by fewer hops.

**Goal**: an F node whose region contains q_goal's arm configuration and whose interval contains α_goal.

**Pruning (exact dominance)**: skip a node if an already-expanded node with the same key (same polytope;
for C also the same entry range and phase) had an interval containing this one at no higher cost.

#### 5a — Node class, heuristic and dominance check

In [30]:
import heapq
from dataclasses import dataclass

HOP_COST = 1e-3   # μ: price of changing polytope (transit / enter). << 1 means "fewest actions first".
TOL = 1e-6


@dataclass
class ReachNode:
    kind: str                  # "F": transit in free region set_id, "C": in grasp component set_id
    set_id: int
    alpha_lo: float            # reachable cap angles in this node: [alpha_lo, alpha_hi]
    alpha_hi: float
    g: float = 0.0             # cost so far: #grasps + #ungrasps + HOP_COST * #hops
    h: float = 0.0             # lower bound on the remaining #actions
    parent: "ReachNode" = None
    action: tuple = ("start",)
    entry_w: float = None      # C only: highest wrist value of (entry region ∩ G)
    entry_w_lo: float = None   # C only: lowest wrist value of (entry region ∩ G)
    phase: str = None          # C only: "entered", "holding" or "released"

    @property
    def f(self):
        return self.g + self.h

    def key(self):
        if self.kind == "F":
            return ("F", self.set_id)
        return ("C", self.set_id, round(self.entry_w, 6), round(self.entry_w_lo, 6), self.phase)

    def __repr__(self):
        name = f"F{self.set_id}" if self.kind == "F" else f"C{self.set_id}"
        phase = f" [{self.phase}]" if self.kind == "C" else ""
        return f"{name:<4} α∈[{self.alpha_lo:+.3f}, {self.alpha_hi:+.3f}]  g={self.g:.3f}{phase}"


def actions_lower_bound(node, alpha_goal):
    remaining = node.alpha_lo - alpha_goal
    n_strokes = 0 if remaining <= TOL else int(np.ceil((remaining - TOL) / L_MAX))
    pending_release = 1.0 if node.kind == "C" and node.phase == "holding" else 0.0
    return 2.0 * n_strokes + pending_release


def dominated(node, expanded):
    return any(lo <= node.alpha_lo + TOL and hi >= node.alpha_hi - TOL and g <= node.g + TOL
               for lo, hi, g in expanded.get(node.key(), []))


free_near_comp = {c: [i for i in range(n_free) if (i, c) in fc_links] for c in range(len(components))}

#### 5b — Successors (the four moves) and the A* loop
Each successor is `(child fields, edge cost, action)`. The action records which polytopes/pieces were
used, so step 6 can place the actual configurations on the chosen sequence.

In [31]:
def successors(node, alpha_goal):
    if node.kind == "F":
        i = node.set_id
        for j in free_neighbors[i]:                                       # transit
            yield (dict(kind="F", set_id=j, alpha_lo=node.alpha_lo, alpha_hi=node.alpha_hi),
                   HOP_COST, ("transit", i, j))
        for c in comps_near[i]:                                           # enter grasp component
            link = max(fc_links[(i, c)], key=lambda l: l["entry_w"])
            yield (dict(kind="C", set_id=c, alpha_lo=node.alpha_lo, alpha_hi=node.alpha_hi,
                        entry_w=link["entry_w"], entry_w_lo=link["exit_w"], phase="entered"),
                   HOP_COST, ("enter", i, c, link["piece"]))
        return

    c, comp = node.set_id, components[node.set_id]
    same = dict(kind="C", set_id=c, alpha_hi=node.alpha_hi, entry_w=node.entry_w, entry_w_lo=node.entry_w_lo)

    if node.phase in ("entered", "released"):                             # stroke = grasp
        reach = comp["w_max"] - node.entry_w_lo if node.phase == "entered" else comp["L"]
        new_lo = max(node.alpha_lo - reach, alpha_goal, cap_lower)
        if new_lo < node.alpha_lo - TOL:
            yield dict(same, alpha_lo=new_lo, phase="holding"), 1.0, ("stroke", c)

    if node.phase in ("entered", "holding"):                              # return = ungrasp
        yield dict(same, alpha_lo=node.alpha_lo, phase="released"), 1.0, ("return", c)

        for j in free_near_comp[c]:                                       # exit
            link = min(fc_links[(j, c)], key=lambda l: l["exit_w"])
            d = max(0.0, link["exit_w"] - node.entry_w)                   # forced forward travel = forced turning
            if node.phase == "entered" and d > TOL:
                continue                                                  # reaching that exit needs a stroke
            new_hi = node.alpha_hi - d
            if new_hi < node.alpha_lo - TOL:
                continue
            cost = 1.0 if node.phase == "holding" else HOP_COST          # releasing the cap is an ungrasp
            yield (dict(kind="F", set_id=j, alpha_lo=node.alpha_lo, alpha_hi=new_hi),
                   cost, ("exit", c, j, link["piece"]))


def reachability_astar(q_init, q_goal, max_expansions=200_000):
    alpha_init, alpha_goal = float(q_init[idx_cap]), float(q_goal[idx_cap])
    if alpha_goal > alpha_init + TOL:
        raise ValueError(f"α_goal={alpha_goal:.3f} > α_init={alpha_init:.3f}: the cap only turns one way (α decreases)")
    if alpha_goal < cap_lower - TOL:
        raise ValueError(f"α_goal={alpha_goal:.3f} is below the cap joint limit {cap_lower:.3f}")

    x_init, x_goal = to_arm(q_init), to_arm(q_goal)
    starts = [i for i, F in enumerate(free_sets) if F.PointInSet(x_init)]
    goal_sets = {i for i, F in enumerate(free_sets) if F.PointInSet(x_goal)}
    if not starts:
        raise ValueError("q_init's arm configuration is in no (cap-dropped) free region")
    if not goal_sets:
        raise ValueError("q_goal's arm configuration is in no (cap-dropped) free region")

    counter = itertools.count()
    open_heap = []
    for i in starts:
        node = ReachNode("F", i, alpha_init, alpha_init)
        node.h = actions_lower_bound(node, alpha_goal)
        heapq.heappush(open_heap, (node.f, next(counter), node))

    expanded = {}      # node key -> list of (alpha_lo, alpha_hi, g) already expanded
    n_expanded = 0
    while open_heap and n_expanded < max_expansions:
        _, _, node = heapq.heappop(open_heap)
        if dominated(node, expanded):
            continue
        expanded.setdefault(node.key(), []).append((node.alpha_lo, node.alpha_hi, node.g))
        n_expanded += 1

        if (node.kind == "F" and node.set_id in goal_sets
                and node.alpha_lo <= alpha_goal + TOL and alpha_goal <= node.alpha_hi + TOL):
            path = []
            while node is not None:
                path.append(node)
                node = node.parent
            path.reverse()
            return path, dict(expanded=n_expanded, starts=starts, goal_sets=sorted(goal_sets))

        for fields, cost, action in successors(node, alpha_goal):
            child = ReachNode(**fields, g=node.g + cost, parent=node, action=action)
            child.h = actions_lower_bound(child, alpha_goal)
            if not dominated(child, expanded):
                heapq.heappush(open_heap, (child.f, next(counter), child))

    return None, dict(expanded=n_expanded, starts=starts, goal_sets=sorted(goal_sets))


def count_actions(path):
    strokes = sum(n.action[0] == "stroke" for n in path)
    returns = sum(n.action[0] == "return" for n in path)
    releases = sum(n.action[0] == "exit" and n.parent.phase == "holding" for n in path)
    hops = sum(n.action[0] in ("transit", "enter") for n in path)
    return dict(grasps=strokes, returns=returns, releases=releases, hops=hops)

#### 5c — Run it on the brute-force scenario
Same start/goal as the brute force (the free-region centroids at both ends of its largest segment, cap
3.0 → −1.0), falling back to q0 if those cells have not been run. With one component of L = 2.0 and
D = 4.0 we expect 2 strokes.

In [32]:
if "lower_poly_centroid" in globals() and "upper_poly_centroid" in globals():
    q_init_astar, q_goal_astar = lower_poly_centroid.copy(), upper_poly_centroid.copy()
else:
    q_init_astar, q_goal_astar = q0.copy(), q0.copy()
    
# define random initial and goal configurations within joint limits, and check that they belong to at least one free polytope
q_init_astar = np.random.uniform(plant.GetPositionLowerLimits(), plant.GetPositionUpperLimits())
q_goal_astar = np.random.uniform(plant.GetPositionLowerLimits(), plant.GetPositionUpperLimits())

while not any(F.PointInSet(to_arm(q_init_astar)) for F in free_sets):
    q_init_astar = np.random.uniform(plant.GetPositionLowerLimits(), plant.GetPositionUpperLimits())
    
while not any(F.PointInSet(to_arm(q_goal_astar)) for F in free_sets):
    q_goal_astar = np.random.uniform(plant.GetPositionLowerLimits(), plant.GetPositionUpperLimits())

q_init_astar[idx_cap] = 3.0
q_goal_astar[idx_cap] = -3.0

t_start = time.time()
astar_path, astar_info = reachability_astar(q_init_astar, q_goal_astar)
print(f"search time {time.time() - t_start:.3f}s, {astar_info['expanded']} nodes expanded")
print(f"start regions {astar_info['starts']}, goal regions {astar_info['goal_sets']}")

if astar_path is None:
    print("No plan found.")
else:
    n = count_actions(astar_path)
    print(f"plan: {n['grasps']} grasp(s) (strokes), {n['returns'] + n['releases']} ungrasp(s) "
          f"({n['returns']} return(s), {n['releases']} release(s)), {n['hops']} hop(s); "
          f"cost {astar_path[-1].g:.3f}\n")
    for node in astar_path:
        print(f"  {str(node.action):<24} -> {node}")

search time 0.086s, 370 nodes expanded
start regions [8, 32], goal regions [0, 4, 5, 12, 13, 19, 20, 23, 40]
plan: 8 grasp(s) (strokes), 8 ungrasp(s) (7 return(s), 1 release(s)), 2 hop(s); cost 16.002

  ('start',)               -> F8   α∈[+3.000, +3.000]  g=0.000
  ('transit', 8, 2)        -> F2   α∈[+3.000, +3.000]  g=0.001
  ('enter', 2, 3, 6)       -> C3   α∈[+3.000, +3.000]  g=0.002 [entered]
  ('stroke', 3)            -> C3   α∈[+2.192, +3.000]  g=1.002 [holding]
  ('return', 3)            -> C3   α∈[+2.192, +3.000]  g=2.002 [released]
  ('stroke', 3)            -> C3   α∈[+1.384, +3.000]  g=3.002 [holding]
  ('return', 3)            -> C3   α∈[+1.384, +3.000]  g=4.002 [released]
  ('stroke', 3)            -> C3   α∈[+0.576, +3.000]  g=5.002 [holding]
  ('return', 3)            -> C3   α∈[+0.576, +3.000]  g=6.002 [released]
  ('stroke', 3)            -> C3   α∈[-0.232, +3.000]  g=7.002 [holding]
  ('return', 3)            -> C3   α∈[-0.232, +3.000]  g=8.002 [released]
  ('stroke'

### Step 6 — Place configurations along the chosen polytope sequence

A* returned *which* polytopes to go through and the exact sequence of grasp actions in each grasp visit
(e.g. `stroke, return, stroke, exit`), with the reachable α intervals. Now we pick actual configurations
(CASSR's second stage), without any optimisation solver, following that action sequence exactly:

1. **Allocate the turning.** Each stroke can turn at most its cap: `w_max − e_lo` for a stroke made straight
   after entering, `L` after a return. Walk the grasp visits backwards from α_goal: each visit turns
   `T = min(α_hi before the visit, α_after + Σ caps) − α_after` (lands exactly on α_init, since the search
   guaranteed α_goal was reachable). T is split among the visit's strokes by water-filling under the caps.
2. **Place points ("ends high" rule, so the exit is always reachable).**
   - transit F_i → F_j: the door (Chebyshev centre of F_i ∩ F_j);
   - enter: if the visit starts with a stroke, the point of F_i ∩ G at wrist `w_max − t1` (clipped to the
     entry range), so the stroke starts right there; if it starts with a return, the highest-wrist point;
   - stroke: wrist ↑ by t, cap −= t;
   - return: wrist ↓ to `min(current, w_max − t_next)`, cap still;
   - exit (release): the point of G ∩ F_j at wrist `min(current, highest exit wrist)` (wrist ↓ or still);
   - finally q_goal.

   Grasp points lie on the segment between G's lowest- and highest-wrist points (or between the extreme
   points of F_i ∩ G for entry/exit), so they stay inside G by convexity.

Each waypoint records the polytope that certifies the straight line *into* it. α goes in the cap slot,
so every waypoint is a full 10-D configuration.

In [33]:
def point_at_wrist(comp, w):
    # Point of the grasp piece with wrist value w, on the segment between its extreme-wrist points.
    lo, hi = comp["chain"][0], comp["chain"][-1]
    s = (w - comp["w_min"]) / comp["L"]
    return lo + np.clip(s, 0.0, 1.0) * (hi - lo)


def point_in_link(link, w):
    # Point of (free region ∩ G) with wrist value w, between its lowest- and highest-wrist points.
    span = link["entry_w"] - link["exit_w"]
    s = 0.0 if span < 1e-12 else (w - link["exit_w"]) / span
    return link["exit"] + np.clip(s, 0.0, 1.0) * (link["entry"] - link["exit"])


def find_link(i, c, piece):
    return next(link for link in fc_links[(i, c)] if link["piece"] == piece)


def split_turn(T, caps):
    # Water-filling: split T as evenly as possible with amounts[i] <= caps[i].
    amounts, remaining = [0.0] * len(caps), T
    for rank, i in enumerate(sorted(range(len(caps)), key=lambda i: caps[i])):
        amounts[i] = min(caps[i], remaining / (len(caps) - rank))
        remaining -= amounts[i]
    assert remaining < 1e-6, f"cannot turn {T:.3f} with stroke caps {caps}"
    return amounts


def build_waypoints(path, q_init, q_goal):
    alpha_init, alpha_goal = float(q_init[idx_cap]), float(q_goal[idx_cap])

    # 1. Split the path into grasp visits (their action lists) and allocate the turning (backward pass).
    visits = []
    for idx, node in enumerate(path):
        a = node.action
        if a[0] == "enter":
            assert len(components[a[2]]["pieces"]) == 1, "step 6 needs MERGE_PIECES = False"
            visits.append(dict(comp=a[2], entry=find_link(a[1], a[2], a[3]), actions=[],
                               hi_before=path[idx - 1].alpha_hi))
        elif a[0] in ("stroke", "return"):
            visits[-1]["actions"].append(a[0])
        elif a[0] == "exit":
            visits[-1]["exit"] = find_link(a[2], a[1], a[3])

    for v in visits:
        comp = components[v["comp"]]
        v["caps"] = [comp["w_max"] - v["entry"]["exit_w"] if pos == 0 else comp["L"]
                     for pos, act in enumerate(v["actions"]) if act == "stroke"]

    alpha_after = alpha_goal
    for v in reversed(visits):
        alpha_before = min(v["hi_before"], alpha_after + sum(v["caps"]))
        v["T"] = alpha_before - alpha_after
        v["amounts"] = split_turn(v["T"], v["caps"]) if v["caps"] else []
        alpha_after = alpha_before
    assert abs(alpha_after - alpha_init) < 1e-6, f"turning allocation failed: {alpha_after} vs {alpha_init}"

    # 2. Place the waypoints: (10-D q, label, certifying set) where the set contains the line into q.
    waypoints = [(np.asarray(q_init, dtype=float).copy(), "start", ("F", path[0].set_id))]
    alpha = alpha_init

    def add(x, label, cert):
        waypoints.append((to_full(x, alpha), label, cert))

    visit_iter = iter(visits)
    for node in path[1:]:
        a = node.action
        if a[0] == "transit":
            add(ff_door[(a[1], a[2])], "transit", ("F", a[1]))
        elif a[0] == "enter":
            v = next(visit_iter)
            comp, cert = components[v["comp"]], ("G", v["comp"])
            e_lo, e_hi = v["entry"]["exit_w"], v["entry"]["entry_w"]
            x_lo, x_hi = v["exit"]["exit_w"], v["exit"]["entry_w"]
            amounts = v["amounts"]
            if v["actions"] and v["actions"][0] == "stroke":
                c = float(np.clip(comp["w_max"] - amounts[0], max(e_lo, x_lo - v["T"]), e_hi))
            else:
                c = e_hi
            add(point_in_link(v["entry"], c), "enter grasp", ("F", a[1]))

            k = 0                                          # index of the next stroke
            for act in v["actions"]:
                if act == "stroke":
                    alpha -= amounts[k]
                    c += amounts[k]
                    k += 1
                    add(point_at_wrist(comp, c), "stroke (grasp)", cert)
                else:
                    t_next = amounts[k] if k < len(amounts) else 0.0
                    c = min(c, comp["w_max"] - t_next)
                    add(point_at_wrist(comp, c), "return (ungrasp)", cert)
            assert c >= x_lo - 1e-6, f"exit wrist {x_lo:.3f} not reachable from {c:.3f}"
            c = min(c, x_hi)
            label = "exit (release)" if v["amounts"] else "exit"
            add(point_in_link(v["exit"], c), label, cert)
        # "stroke", "return" and "exit" were already placed while executing the visit.

    add(to_arm(q_goal), "goal", ("F", path[-1].set_id))
    assert abs(alpha - alpha_goal) < 1e-6
    return waypoints, visits

astar_waypoints, astar_visits = build_waypoints(astar_path, q_init_astar, q_goal_astar)
for v in astar_visits:
    print(f"grasp visit in C{v['comp']}: actions {v['actions']}, turns T = {v['T']:.3f} rad "
          f"as {np.round(v['amounts'], 3).tolist()}")
print(f"\n{len(astar_waypoints)} waypoints:")
for q, label, cert in astar_waypoints:
    print(f"  {label:<18} in {cert[0]}{cert[1]:<3} wrist {q[idx_wrist]:+.3f}  cap {q[idx_cap]:+.3f}")

grasp visit in C3: actions ['stroke', 'return', 'stroke', 'return', 'stroke', 'return', 'stroke', 'return', 'stroke', 'return', 'stroke', 'return', 'stroke', 'return', 'stroke'], turns T = 6.000 rad as [0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75]

20 waypoints:
  start              in F8   wrist +0.170  cap +3.000
  transit            in F8   wrist +0.978  cap +3.000
  enter grasp        in F2   wrist -0.942  cap +3.000
  stroke (grasp)     in G3   wrist -0.192  cap +2.250
  return (ungrasp)   in G3   wrist -0.942  cap +2.250
  stroke (grasp)     in G3   wrist -0.192  cap +1.500
  return (ungrasp)   in G3   wrist -0.942  cap +1.500
  stroke (grasp)     in G3   wrist -0.192  cap +0.750
  return (ungrasp)   in G3   wrist -0.942  cap +0.750
  stroke (grasp)     in G3   wrist -0.192  cap +0.000
  return (ungrasp)   in G3   wrist -0.942  cap +0.000
  stroke (grasp)     in G3   wrist -0.192  cap -0.750
  return (ungrasp)   in G3   wrist -0.942  cap -0.750
  stroke (grasp)     in G3   wri

#### 6b — Check every straight-line move
Each move must (1) have both endpoints in its certifying polytope (so the whole line is inside it) and
(2) respect the cap rule: inside a grasp piece, wrist ↑ turns the cap by exactly −Δwrist, wrist ↓ keeps
it; outside grasp the cap never moves.

In [34]:
def certifying_set(cert):
    kind, idx = cert
    return free_sets[idx] if kind == "F" else pieces[components[idx]["pieces"][0]]["poly"]


def check_waypoints(waypoints, verbose=True, tol=1e-6):
    """Every move has both endpoints in its certifying set and respects the cap rule."""
    all_ok = True
    for (qa, _, _), (qb, label, cert) in zip(waypoints[:-1], waypoints[1:]):
        poly = certifying_set(cert)
        inside = poly.PointInSet(to_arm(qa), tol) and poly.PointInSet(to_arm(qb), tol)
        dw, dcap = qb[idx_wrist] - qa[idx_wrist], qb[idx_cap] - qa[idx_cap]
        if cert[0] == "G" and dw > TOL:
            cap_ok = abs(dcap + dw) < tol
        else:
            cap_ok = abs(dcap) < tol
        all_ok &= inside and cap_ok
        if verbose:
            print(f"  {label:<20} {cert[0]}{cert[1]:<3} inside={inside!s:<5} cap_rule={cap_ok!s:<5} "
                  f"Δwrist={dw:+.3f} Δcap={dcap:+.3f}")
    if verbose:
        print("all moves valid:", all_ok)
    return all_ok


check_waypoints(astar_waypoints)

  transit              F8   inside=True  cap_rule=True  Δwrist=+0.808 Δcap=+0.000
  enter grasp          F2   inside=True  cap_rule=True  Δwrist=-1.919 Δcap=+0.000
  stroke (grasp)       G3   inside=True  cap_rule=True  Δwrist=+0.750 Δcap=-0.750
  return (ungrasp)     G3   inside=True  cap_rule=True  Δwrist=-0.750 Δcap=+0.000
  stroke (grasp)       G3   inside=True  cap_rule=True  Δwrist=+0.750 Δcap=-0.750
  return (ungrasp)     G3   inside=True  cap_rule=True  Δwrist=-0.750 Δcap=+0.000
  stroke (grasp)       G3   inside=True  cap_rule=True  Δwrist=+0.750 Δcap=-0.750
  return (ungrasp)     G3   inside=True  cap_rule=True  Δwrist=-0.750 Δcap=+0.000
  stroke (grasp)       G3   inside=True  cap_rule=True  Δwrist=+0.750 Δcap=-0.750
  return (ungrasp)     G3   inside=True  cap_rule=True  Δwrist=-0.750 Δcap=+0.000
  stroke (grasp)       G3   inside=True  cap_rule=True  Δwrist=+0.750 Δcap=-0.750
  return (ungrasp)     G3   inside=True  cap_rule=True  Δwrist=-0.750 Δcap=+0.000
  stroke (grasp)

np.True_

### Step 7 — Show the plan in Meshcat
Same playback as the brute force: straight-line interpolation between consecutive waypoints (the cap
value is interpolated too, which is exactly the α −= Δwrist rule on forward runs). The trajectory is
also stored as a `PiecewisePolynomial`, so it can be given to the slice viewer (section 5).

**Markers** (`meshcat_plan_markers.py`): green = start, orange = goal.
- Translucent ghosts of the gripper (its real meshes) at the start and goal configurations.
- On the cap: a green and an orange tick at the start and goal angles. The cap is at the start (goal)
  when its red line is on the green (orange) tick. The grey spiral arc and its arrowhead show
  which way and how far the cap turns, so a turn close to 2π is not confused with a small one.
- Around the wrist axis: the joint range (grey), the angles outside it (red), green and orange ticks
  at the start and goal wrist values, and a black needle along the finger opening that turns with
  the wrist. The needle staying inside the grey band shows that the plan respects the wrist limits.

In [35]:
from pydrake.trajectories import PiecewisePolynomial
import meshcat_plan_markers
import importlib
importlib.reload(meshcat_plan_markers)   # for iterative development in a notebook
from meshcat_plan_markers import PlanMarkers

def play_waypoints(waypoints, step=0.02, pause=0.02, markers=None):
    for (qa, _, _), (qb, label, _) in zip(waypoints[:-1], waypoints[1:]):
        n = max(2, int(np.ceil(np.linalg.norm(qb - qa) / step)))
        for s in np.linspace(0.0, 1.0, n):
            plant.SetPositions(plant_context, qa + s * (qb - qa))
            if markers is not None:
                markers.update(plant_context)
            diagram.ForcedPublish(diagram_context)
            time.sleep(pause)


Q = np.array([q for q, _, _ in astar_waypoints])
times = np.concatenate([[0.0], np.cumsum(np.linalg.norm(np.diff(Q, axis=0), axis=1))])
keep = np.concatenate([[True], np.diff(times) > 1e-9])        # drop zero-length moves
astar_traj = PiecewisePolynomial.FirstOrderHold(times[keep], Q[keep].T)
print(f"path length (10-D, cap included): {times[-1]:.3f}")

wrist_joint = plant.GetJointByName("panda_joint7", panda_arm)
w_lo, w_hi = wrist_joint.position_lower_limits()[0], wrist_joint.position_upper_limits()[0]
print("wrist within its limits at every waypoint:",
      all(w_lo - 1e-9 <= q[idx_wrist] <= w_hi + 1e-9 for q, _, _ in astar_waypoints))

markers = PlanMarkers(visualizer.meshcat_task_space, plant, visualizer.model_inspector, panda_hand,
                      wrist_joint=wrist_joint,
                      cap_joint=plant.GetJointByName("cap_to_base", cap))
markers.set_start_goal(q_init_astar, q_goal_astar)

play_waypoints(astar_waypoints, markers=markers)

path length (10-D, cap included): 31.942
wrist within its limits at every waypoint: True


In [36]:
# Optional: overlay the plan on the wrist/cap slice viewer (needs section 5's slice_viz)
if "slice_viz" in globals():
    slice_viz.set_trajectory(astar_traj)

## GCS baseline on the same polytopes

Same problem as the reachability A*, solved with Drake's `GcsTrajectoryOptimization` on the **same
sets and the same graph**: the cap-dropped free regions `free_sets` and the grasp pieces of
`components` (Steps 1–3), connected by the A*'s own `ff_door` / `fc_links` (Step 4). Each set is lifted
back to 10-D with the cap free inside its joint limits.

**Two modes, as linear constraints on each vertex's Bézier control points** r₀ → r₁ (order 1, so
every move is a straight line with both ends in one convex set, the same certificate as the A*
waypoints). They are the placement and grasp constraints of `ManipulationPlanner._solve_for_n_grasps_CC`,
attached to vertices instead of point indices:
- free vertex (transit, return, approach): r₁[cap] = r₀[cap], so the cap is frozen;
- grasp vertex (stroke): r₁[cap] + r₁[wrist] = r₀[cap] + r₀[wrist], so α −= Δwrist, and
  r₁[wrist] ≥ r₀[wrist] (forward only, the A* ratchet).

**Layers.** A GCS path visits each vertex at most once, but several strokes reuse the same grasp piece,
with a return in between. So the graph is unrolled by the number of grasps:
`approach → grasp1 → between1 → grasp2 → … → graspN → depart`. Every grasp layer can also go straight
to `depart`, and `approach` can go straight to the goal (no grasp). The big-M binaries of the MIQP
become GCS edge flows, and its loop over the number of grasps becomes the layers.
- `between` layers hold only the free regions that touch a grasp piece (`GRASP_NEAR`), where returns
  and regrasps happen.
- `approach` holds the start regions + `GRASP_NEAR`, `depart` holds `GRASP_NEAR` + the goal regions
  (`PRUNE_FREE_LAYERS = True`). 850 of the 861 pairs of free regions intersect, so every free layer
  with all 42 regions adds ~1700 edges, and the relaxation needs ~2–6 MB per edge: full layers
  were killed at 35–59 GB. Pruning only rules out detours through regions that touch neither the
  start, the goal nor a grasp piece. `keep=` adds regions back (the A* path's, for the restriction).

**Cost**, matching the A*: `GRASP_COST` on every free → grasp edge (one grasp + its release),
`HOP_COST` on every other polytope change, plus the path length. With `GRASP_COST` ≫ path length the
fewest grasps win first.

`SolvePath` solves the convex relaxation, rounds it into up to `max_rounded_paths` candidate polytope
sequences, and solves a convex restriction (the trajectory for a fixed sequence) for each. To split the
time, the restriction on the chosen sequence is solved again on its own; set `TIME_RELAXATION = True` to
also solve the relaxation alone (its cost is a lower bound).

In [37]:
from pydrake.planning import GcsTrajectoryOptimization
from pydrake.geometry.optimization import GraphOfConvexSetsOptions, Point
from pydrake.solvers import MosekSolver

GRASP_COST = 10.0      # per grasp (stroke + release): fewest grasps first, like the A*
GCS_ORDER = 1          # straight segments, same path class as the A* waypoints

# MOSEK reads its licence from MOSEKLM_LICENSE_FILE, which only the conda env's activate script sets.
# A kernel started without activating the env would silently fall back to another solver.
os.environ.setdefault("MOSEKLM_LICENSE_FILE", str(REPO_ROOT / "mosek.lic"))
print("Mosek enabled:", MosekSolver().enabled(), "(otherwise Drake falls back to another convex solver, and no exact MIP)")


def lift_cap(poly9):
    """9-D set (cap dropped) -> 10-D set with the cap free within its joint limits."""
    A = np.insert(poly9.A(), idx_cap, 0.0, axis=1)
    e_cap = np.zeros(A.shape[1])
    e_cap[idx_cap] = 1.0
    return HPolyhedron(np.vstack([A, e_cap, -e_cap]), np.concatenate([poly9.b(), [cap_upper, -cap_lower]]))


free_regions = [lift_cap(F) for F in free_sets]
grasp_regions = [lift_cap(pieces[comp["pieces"][0]]["poly"]) for comp in components]


def freeze_cap(subgraph):
    r = subgraph.vertex_control_points()
    for j in range(1, r.shape[1]):
        subgraph.AddVertexConstraint(r[idx_cap, j] == r[idx_cap, 0])


def turn_cap_with_wrist(subgraph):
    r = subgraph.vertex_control_points()
    for j in range(1, r.shape[1]):
        subgraph.AddVertexConstraint(r[idx_cap, j] + r[idx_wrist, j] == r[idx_cap, 0] + r[idx_wrist, 0])
        subgraph.AddVertexConstraint(r[idx_wrist, j] >= r[idx_wrist, j - 1])


def add_constant_edge_cost(edges, cost):
    # A constant cost needs a placeholder variable: 0 * (duration of the edge's first vertex)
    edges.AddEdgeCost(cost + 0 * edges.edge_constituent_vertex_durations()[0])


GRASP_NEAR = sorted({i for i, _ in fc_links})   # free regions that touch a grasp piece
FULL_BETWEEN_LAYERS = False                      # True: every between layer holds all free regions (much bigger)
PRUNE_FREE_LAYERS = True                         # False: approach/depart hold all free regions (does not fit in memory)


def add_free_layer(gcs, name, region_ids):
    """Subgraph of the given free regions (global ids), with the A*'s transit edges among them."""
    local = {i: k for k, i in enumerate(region_ids)}
    edges = [(local[i], local[j]) for (i, j) in ff_door if i in local and j in local]
    F = gcs.AddRegions([free_regions[i] for i in region_ids], edges, order=GCS_ORDER, name=name)
    freeze_cap(F)
    add_constant_edge_cost(F, HOP_COST)
    return dict(subgraph=F, ids=list(region_ids), local=local)


def connect_free_grasp(gcs, layer, G, free_to_grasp):
    local = layer["local"]
    pairs = [(local[i], c) for i, c in fc_links if i in local]
    if free_to_grasp:
        add_constant_edge_cost(gcs.AddEdges(layer["subgraph"], G, edges_between_regions=pairs), GRASP_COST)
    else:
        add_constant_edge_cost(gcs.AddEdges(G, layer["subgraph"],
                                            edges_between_regions=[(c, k) for k, c in pairs]), HOP_COST)


def build_layered_gcs(q_init, q_goal, n_layers, keep=()):
    """keep: extra free regions for the approach/depart layers (e.g. the ones the A* path uses)."""
    gcs = GcsTrajectoryOptimization(plant.num_positions())
    x_init, x_goal = to_arm(q_init), to_arm(q_goal)
    all_ids = list(range(len(free_sets)))
    starts = [i for i in all_ids if free_sets[i].PointInSet(x_init)]
    goals = [i for i in all_ids if free_sets[i].PointInSet(x_goal)]
    between_ids = all_ids if FULL_BETWEEN_LAYERS else GRASP_NEAR
    if PRUNE_FREE_LAYERS:
        approach_ids = sorted(set(starts) | set(GRASP_NEAR) | set(keep))
        depart_ids = sorted(set(GRASP_NEAR) | set(goals) | set(keep))
    else:
        approach_ids = depart_ids = all_ids

    approach = add_free_layer(gcs, "approach", approach_ids)
    depart = add_free_layer(gcs, "depart", depart_ids)
    between, grasp = [], []
    for n in range(1, n_layers + 1):
        G = gcs.AddRegions(grasp_regions, [], order=GCS_ORDER, name=f"grasp{n}")
        turn_cap_with_wrist(G)
        connect_free_grasp(gcs, approach if n == 1 else between[-1], G, free_to_grasp=True)
        connect_free_grasp(gcs, depart, G, free_to_grasp=False)
        if n < n_layers:
            between.append(add_free_layer(gcs, f"between{n}", between_ids))
            connect_free_grasp(gcs, between[-1], G, free_to_grasp=False)
        grasp.append(G)

    source = gcs.AddRegions([Point(q_init)], order=0, name="source")
    target = gcs.AddRegions([Point(q_goal)], order=0, name="target")
    gcs.AddEdges(source, approach["subgraph"], edges_between_regions=[
        (0, approach["local"][i]) for i in starts if i in approach["local"]])
    for layer in (approach, depart):
        gcs.AddEdges(layer["subgraph"], target, edges_between_regions=[
            (layer["local"][i], 0) for i in goals if i in layer["local"]])
    gcs.AddPathLengthCost()

    info = {source.Vertices()[0].id(): ("source", 0, 0), target.Vertices()[0].id(): ("target", 0, 0)}
    for name, layer in [("approach", approach), ("depart", depart)] + [(f"between{n}", l) for n, l in enumerate(between, 1)]:
        info.update({v.id(): ("F", name, i) for v, i in zip(layer["subgraph"].Vertices(), layer["ids"])})
    for n, G in enumerate(grasp, start=1):
        info.update({v.id(): ("G", n, c) for c, v in enumerate(G.Vertices())})
    return dict(gcs=gcs, source=source, target=target, approach=approach, between=between,
                depart=depart, grasp=grasp, info=info)


def solved_vertex_sequence(built, result):
    """Vertices on the solution path, source to target (edges with flow ~1)."""
    graph = built["gcs"].graph_of_convex_sets()
    next_vertex = {e.u().id(): e.v() for e in graph.Edges() if result.GetSolution(e.phi()) > 0.5}
    seq = [built["source"].Vertices()[0]]
    while seq[-1].id() in next_vertex and len(seq) <= graph.num_vertices():
        seq.append(next_vertex[seq[-1].id()])
    return seq


def vertex_control_points(v, result):
    """Control points (nq x order+1) of a vertex's segment: its solution is [control points (column-major), duration]."""
    x = v.GetSolution(result)
    nq = plant.num_positions()
    return x[: nq * ((len(x) - 1) // nq)].reshape((nq, -1), order="F")


def gcs_waypoints(built, result, seq):
    """GCS solution -> A*-style (q, label, cert) waypoints: one straight move per vertex (order 1).
    Read from the vertex solutions, not from the returned trajectory, which can drop zero-duration segments."""
    waypoints = []
    for i, v in enumerate(seq):
        kind, layer, idx = built["info"][v.id()]
        control_points = vertex_control_points(v, result)
        if kind == "source":
            first = built["info"][seq[1].id()]
            waypoints.append((control_points[:, 0].copy(), "start", (first[0], first[2])))
        elif kind in ("F", "G"):
            label = f"free ({layer})" if kind == "F" else f"stroke (grasp {layer})"
            waypoints.append((control_points[:, -1].copy(), label, (kind, idx)))
    return waypoints


def plan_metrics(waypoints):
    moves = list(zip(waypoints[:-1], waypoints[1:]))
    strokes = sum(cert[0] == "G" and qb[idx_wrist] - qa[idx_wrist] > TOL for (qa, _, _), (qb, _, cert) in moves)
    certs = [cert for _, (_, _, cert) in moves]
    changes = sum(a != b for a, b in zip(certs[:-1], certs[1:]))
    length = sum(np.linalg.norm(qb - qa) for (qa, _, _), (qb, _, _) in moves)
    return dict(grasps=strokes, polytope_changes=changes, path_length=length)


def collisions_along(waypoints, step=0.01):
    """Samples (every ~step in 10-D) that the collision checker flags, out of all samples."""
    n_hit = n_total = 0
    for (qa, _, _), (qb, _, _) in zip(waypoints[:-1], waypoints[1:]):
        n = max(2, int(np.ceil(np.linalg.norm(qb - qa) / step)))
        for s in np.linspace(0.0, 1.0, n):
            n_hit += bool(visualizer.check_collision_q_by_ik(qa + s * (qb - qa)))
            n_total += 1
    return n_hit, n_total

Mosek enabled: True (otherwise Drake falls back to another convex solver, and no exact MIP)


### Full GCS
The A* section's query turns the cap by 6 rad: 8 grasps, and its full GCS does not fit in 60 GB (paper
Table III, Run E). This section solves a smaller turn, `GCS_TURN`, from the same start, and runs the A* on
the same query, so all plans below compare like with like. Grasp layers: N = ⌈|Δα| / L_max⌉ + 1, the
fewest strokes that could reach the goal plus one spare (independent of the A* result). The free regions
of that A* plan are kept in the approach/depart layers, so the restriction below can use the same graph.

Expected cost on this machine (Run A, licence held): 0.5 rad, 1 grasp: ~4 GB, ~45 s; 1.5 rad, 2 grasps:
~6 GB, ~95 s.

In [ ]:
GCS_TURN = 1.5          # rad; 0.5 -> 1 grasp, 1.5 -> 2 grasps. 6 (the A* section's query) does not fit in memory.
TIME_RELAXATION = False  # True: also solve the relaxation on its own (its cost is a lower bound); roughly doubles the time

q_goal_gcs = q_goal_astar.copy()
q_goal_gcs[idx_cap] = q_init_astar[idx_cap] - GCS_TURN     # the code turns the cap towards negative angles
alpha_init, alpha_goal = float(q_init_astar[idx_cap]), float(q_goal_gcs[idx_cap])
N_LAYERS = int(np.ceil(abs(alpha_goal - alpha_init) / L_MAX)) + 1

# The A* on the same query (the restriction and the comparison below use it)
gcs_astar_path, _ = reachability_astar(q_init_astar, q_goal_gcs)
gcs_astar_waypoints, _ = build_waypoints(gcs_astar_path, q_init_astar, q_goal_gcs)

t0 = time.perf_counter()
built = build_layered_gcs(q_init_astar, q_goal_gcs, N_LAYERS,
                          keep={node.set_id for node in gcs_astar_path if node.kind == "F"})
t_build = time.perf_counter() - t0
graph = built["gcs"].graph_of_convex_sets()
print(f"turn {GCS_TURN} rad, {N_LAYERS} grasp layers: {graph.num_vertices()} vertices, {graph.num_edges()} edges "
      f"(built in {t_build:.2f}s)")

gcs_options = GraphOfConvexSetsOptions()
gcs_options.max_rounded_paths = 10
if TIME_RELAXATION:
    relax_options = GraphOfConvexSetsOptions()
    relax_options.max_rounded_paths = 0             # relaxation only
    t0 = time.perf_counter()
    _, relax_result = built["gcs"].SolvePath(built["source"], built["target"], relax_options)
    print(f"relaxation alone: {time.perf_counter() - t0:.1f}s, lower bound {relax_result.get_optimal_cost():.3f}")

t0 = time.perf_counter()
gcs_traj, gcs_result = built["gcs"].SolvePath(built["source"], built["target"], gcs_options)
t_total = time.perf_counter() - t0
print(f"solver {gcs_result.get_solver_id().name()}, success {gcs_result.is_success()}, "
      f"cost {gcs_result.get_optimal_cost():.3f}, SolvePath {t_total:.1f}s")

gcs_seq = solved_vertex_sequence(built, gcs_result)
t0 = time.perf_counter()
_, gcs_restriction_result = built["gcs"].SolveConvexRestriction(gcs_seq, gcs_options)
t_traj = time.perf_counter() - t0
print(f"trajectory for the chosen sequence alone: {t_traj:.3f}s -> choosing the sequence ≈ {t_total - t_traj:.1f}s")

gcs_full_waypoints = gcs_waypoints(built, gcs_result, gcs_seq)
print(f"\n{len(gcs_full_waypoints)} waypoints:")
for q, label, cert in gcs_full_waypoints:
    print(f"  {label:<18} in {cert[0]}{cert[1]:<3} wrist {q[idx_wrist]:+.3f}  cap {q[idx_cap]:+.3f}")
if not check_waypoints(gcs_full_waypoints, verbose=False):   # print every move only if one fails
    check_waypoints(gcs_full_waypoints)
else:
    print("all moves valid: True")

10 grasp layers: 544 vertices, 19801 edges (built in 2.65s)


### GCS restricted to the A* polytope sequence
The A* path is mapped onto the layered graph, and only the configurations are optimised
(`SolveConvexRestriction`, one convex program). This separates "GCS picks better polytopes" from
"GCS places configurations better in the same polytopes". Free moves go in `approach` before the first
stroke, in `depart` after the last one, and in `between{n}` after stroke n:
- start / transit / exit → the free region the A* is in; enter piece k → free region F_k (it contains
  the piece, so the ungrasped approach and returns happen there);
- stroke → the grasp piece in the next grasp layer; return → back to F_k;
- exit → F_k first (the A* releases and moves ungrasped inside the piece), then F_j.

In [ ]:
def astar_vertex_sequence(built, path):
    n_strokes = sum(node.action[0] == "stroke" for node in path)
    if n_strokes > len(built["grasp"]):
        raise ValueError(f"the A* plan needs {n_strokes} grasp layers, the graph has {len(built['grasp'])}")
    piece_region = lambda c: components[c]["pieces"][0]    # the piece lives in the free region it was cut from

    def free_vertex(n, i):
        layer = built["approach"] if n == 0 else built["depart"] if n == n_strokes else built["between"][n - 1]
        if i not in layer["local"]:
            raise ValueError(f"free region {i} is not in the free layer after stroke {n}")
        return layer["subgraph"].Vertices()[layer["local"][i]]

    n = 0
    seq = [built["source"].Vertices()[0], free_vertex(0, path[0].set_id)]
    for node in path[1:]:
        a = node.action
        if a[0] == "transit":
            v = free_vertex(n, a[2])
        elif a[0] == "enter":
            v = free_vertex(n, a[3])
        elif a[0] == "stroke":
            n += 1
            v = built["grasp"][n - 1].Vertices()[a[1]]
        elif a[0] == "return":
            v = free_vertex(n, piece_region(a[1]))
        elif a[0] == "exit":
            # the A* releases and moves ungrasped inside the piece (⊂ F_k) before leaving to F_j
            release = free_vertex(n, piece_region(a[1]))
            if release.id() != seq[-1].id():
                seq.append(release)
            v = free_vertex(n, a[2])
        else:
            raise ValueError(f"unknown action {a}")
        if v.id() != seq[-1].id():
            seq.append(v)
    seq.append(built["target"].Vertices()[0])

    ids = [v.id() for v in seq]
    if len(set(ids)) != len(ids):
        raise ValueError("the A* sequence visits a polytope twice in one layer; a restriction can't")
    return seq


astar_seq = astar_vertex_sequence(built, gcs_astar_path)
print("A* sequence on the layered graph:", " -> ".join(
    f"{k}{i}" + (f"@{n}" if k in ("F", "G") else "") for k, n, i in (built["info"][v.id()] for v in astar_seq)))

t0 = time.perf_counter()
astar_gcs_traj, astar_gcs_result = built["gcs"].SolveConvexRestriction(astar_seq, gcs_options)
t_astar_gcs = time.perf_counter() - t0
print(f"success {astar_gcs_result.is_success()}, cost {astar_gcs_result.get_optimal_cost():.3f}, time {t_astar_gcs:.3f}s")

gcs_on_astar_waypoints = gcs_waypoints(built, astar_gcs_result, astar_seq)
if not check_waypoints(gcs_on_astar_waypoints, verbose=False):   # print every move only if one fails
    check_waypoints(gcs_on_astar_waypoints)
else:
    print("all moves valid: True")

### Comparison
All three plans solve the same query (`GCS_TURN`). The A* is timed again here, split the same way as the
GCS: search (polytope sequence) and `build_waypoints` (configurations). The shared preprocessing (the A*
graph cells) is used by all methods and not counted. "Colliding samples" counts the collision checker's
hits every ~0.01 rad along each plan; it also counts the fingers touching the cap during strokes, so read
it as a rough check, not a collision rate. For timed, memory-measured runs and side-by-side videos, use
`experiments/gcs_comparison/compare_plans.py`.

In [ ]:
t0 = time.perf_counter()
astar_path_timed, _ = reachability_astar(q_init_astar, q_goal_gcs)
t_astar_search = time.perf_counter() - t0
t0 = time.perf_counter()
build_waypoints(astar_path_timed, q_init_astar, q_goal_gcs)
t_astar_place = time.perf_counter() - t0

rows = [
    ("A*", gcs_astar_waypoints, t_astar_search, t_astar_place),
    ("GCS on A* sequence", gcs_on_astar_waypoints, t_astar_search, t_astar_gcs),
    ("full GCS", gcs_full_waypoints, t_total - t_traj, t_traj),
]
print(f"turn {GCS_TURN} rad")
print(f"{'plan':<20} {'grasps':>6} {'changes':>7} {'length':>8} {'t_seq [s]':>10} {'t_traj [s]':>10} "
      f"{'valid':>6} {'colliding samples':>18}")
for name, waypoints, t_seq, t_trj in rows:
    m = plan_metrics(waypoints)
    hit, total = collisions_along(waypoints)
    print(f"{name:<20} {m['grasps']:>6} {m['polytope_changes']:>7} {m['path_length']:>8.3f} {t_seq:>10.3f} "
          f"{t_trj:>10.3f} {check_waypoints(waypoints, verbose=False)!s:>6} {f'{hit}/{total}':>18}")
print(f"\nfull GCS graph build: {t_build:.3f}s (not included above)")

In [ ]:
# Play a plan in Meshcat with the start/goal and wrist-limit markers (Step 7)
markers.set_start_goal(q_init_astar, q_goal_gcs)
play_waypoints(gcs_full_waypoints, markers=markers)
# play_waypoints(gcs_on_astar_waypoints, markers=markers)
# play_waypoints(gcs_astar_waypoints, markers=markers)

## Test how accurate are the different c-free generated shapes around wrist rotation
I found the polytopes are not accurate in some rotations, we compare how they work against the collision checker.

- Load the different collision-free shapes from data/cfree/
- Get number of false positives, false negatives, true positives, and true negatives for each shape group, along wrist rotation as in the previous section.

In [ ]:
cfree_filenames = ["cfree_10dof_irisnp2_98_k42.yaml", "cfree_10dof_irisnp2_99_k46.yaml", "cfree_97coverage_iriszo.yaml", "cfree_97coverage_iriszo(64 regions).yaml"]

cfree_regions_list = []
for filename in cfree_filenames:
    cfree_regions_dict = LoadIrisRegionsYamlFile(f"data/cfree/{filename}")
    cfree_regions_list.append(list(cfree_regions_dict.values()))
    

for i, cfree_regions in enumerate(cfree_regions_list):
    print(f"\nChecking collision-free segments against c-free regions from {cfree_filenames[i]}...")
    print(f"Number of c-free regions: {len(cfree_regions)}.")
    print(f"Is grasp configuration in any c-free region: {any(poly.PointInSet(grasp_configuration) for poly in cfree_regions)}.")
    
    TP = 0
    FP = 0
    TN = 0
    FN = 0
        
    for i in range(len(wrist_angles) - 1):
        q = grasp_configuration.copy()
        q[idx_wrist] = wrist_angles[i]
        
        true_collision_free = not visualizer.check_collision_q_by_ik(q)
        in_cfree = any(poly.PointInSet(q) for poly in cfree_regions)
        
        if true_collision_free and in_cfree:
            TP += 1
        elif true_collision_free and not in_cfree:
            FN += 1
        elif not true_collision_free and in_cfree:
            FP += 1
        elif not true_collision_free and not in_cfree:
            TN += 1
            
    print(f"True Positives: {TP}, False Positives: {FP}, True Negatives: {TN}, False Negatives: {FN}. Our of {len(wrist_angles)} wrist angles.")
        
        

### Plan a trajectory

`x_goal` reuses `q0` with only the cap angle advanced by `-1.0` rad (roughly
the wrist's own achievable range) - everything else about the pose is
unchanged, since the grasp constraint is exactly insensitive to the cap's
own value (finding 2 in `tc_space_investigation_summary.md` Part 2). This
keeps `x_goal` inside both `grasp_polytope` and `placement_polytope` by
construction, and is achievable within a single grasp cycle, so a sensible
first run should find a short path (small `n_grasps`).

In [ ]:
q_init = np.zeros(plant.num_positions())
q_init[idx_cap] = 0.0
q_init[idx_f1] = -0.0
q_init[idx_f2] = 0.04

q_goal = q_init.copy()
q_goal[idx_cap] = 2.0

print(f"x_init  = {np.round(q_init, 4)}")
print(f"x_goal  = {np.round(q_goal, 4)}")

path, traj = planner.compute_trajectory(q_init, q_goal, display=True)

In [ ]:
planner.plant.SetPositions(planner.plant_context, q_init)
traj_path = planner.display_trajectory(traj, plotly=False)

## 5. C-space slice viewer

Interactive 2-D cross-section of the grasp/placement/c-free polytopes
(`cspace_slice_visualizer.py`) - reuses `visualizer` (this notebook's own
`CIrisPlantVisualizer`) so dragging any slider also moves the Meshcat robot
above. Defaults to the wrist/cap plane (the two DOF the grasp polytope
leaves free - see `tc_space_investigation_summary.md` Part 2), seeded at
`q0` so the first render is non-degenerate.

For exploring the same polytopes *while this notebook's kernel is busy*
solving (section 4), use `cspace_explorer.ipynb` instead, in its own
kernel - this cell's sliders are frozen for as long as any cell above is
still running.

In [ ]:
from cspace_slice_visualizer import CSpaceSliceVisualizer

slice_viz = CSpaceSliceVisualizer.from_visualizer(
    visualizer,
    groups={
        "c-free": (planner.cs_free, "steelblue"),
        "grasp": (grasp_polytope, "seagreen"),
        "placement": (placement_polytope, "darkorange"),
    },
    q0=q0,
    free_axes=(idx_wrist, idx_cap),  # the two DOF the grasp polytope leaves free
)
slice_viz.show()


In [ ]:
# Once compute_trajectory (section 4) has run:
slice_viz.set_trajectory(traj)
# or, if only the sampled path is available (traj_path from display_trajectory):
# slice_viz.set_path(traj_path)
